[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/review_1_python/blob/main/sesiones/refuerzo_python_puro.ipynb)

# 🧱 Refuerzo de Python puro
## Diccionarios, estado acumulativo y comparación de tuplas

Práctica intensiva de **≈ 3 h 40 min** con datos pequeños y realistas (ventas de bodega, movimientos de cuenta, kárdex de stock) en soles y con fechas ISO `AAAA-MM-DD`.
Solo usamos la **biblioteca estándar** de Python (`matplotlib` aparece una única vez, para un gráfico ilustrativo).

### ✅ Objetivos (marca una casilla cuando puedas hacerlo sin mirar)

**Módulo 1 — Diccionarios**
- ☐ Explicar por qué `d[clave]` da `KeyError` y evitarlo con `if clave in d`.
- ☐ Agrupar con el patrón `if / else` (inicializar en el `else`, sumar después).
- ☐ Acumular con `d.get(clave, 0)`.
- ☐ Recorrer con `for k in d`, `.keys()`, `.values()` y `.items()`, sabiendo qué se puede cambiar mientras recorres.
- ☐ Entender por qué aparece `477.5799999999999` y redondear al entregar el resultado.

**Módulo 2 — Estado acumulativo y control de flujo**
- ☐ Distinguir agregación final (`sum`) de seguimiento paso a paso.
- ☐ Arrancar el estado en su valor inicial (no en 0).
- ☐ Detectar eventos dentro del bucle: primero, todos y cruces (con el valor anterior).
- ☐ Buscar extremos a mano con `None` como centinela y guardar el registro completo.
- ☐ Elegir entre `break` y `continue` sin saltarte casos válidos.

**Módulo 3 — Tuplas y `key`**
- ☐ Predecir comparaciones de tuplas y de textos.
- ☐ Evitar la trampa de `min(lista_de_tuplas)`.
- ☐ Usar `key=` en `min`, `max` y `sorted`, incluido el desempate con una tupla.
- ☐ Combinar `zip` y `enumerate`.

**Reto final**
- ☐ Resolver un problema completo primero a tu manera y luego en **un solo recorrido**.

### 🧭 Cómo usar este notebook

1. **Guarda una copia en tu Drive** (`Archivo → Guardar una copia en Drive`) para no perder tu trabajo.
2. **Ejecuta las celdas en orden**, de arriba abajo.
3. **Intenta antes de abrir pistas.** Cada ejercicio trae 1 o 2 celdas «💡 Pista» que aparecen colapsadas: haz doble clic o pulsa *Mostrar código* solo si estás atascado. Las **🔒 Soluciones** están al final de cada módulo.
4. Las celdas **🔮 Predice antes de ejecutar** tienen comentarios donde escribes tu predicción. Escríbela *antes* de ejecutar: equivocarte ahí es lo que más enseña.
5. Cada ejercicio es independiente: define sus propios datos en la misma celda. Al terminar tu código, ejecuta la celda de validación; si falla, el mensaje te apunta al error típico.

| Bloque | Tiempo |
|---|---|
| Portada | 5 min |
| Módulo 1 — Diccionarios: agrupación y acumulación | 50 min |
| Módulo 2 — Estado acumulativo y control de flujo | 70 min |
| Módulo 3 — Comparación de tuplas y parámetro `key` | 50 min |
| Reto final integrador | 40 min |
| Cierre | 5 min |
| **Total** | **≈ 3 h 40 min** |

### 🎯 Errores típicos que vas a cazar hoy

1. **Arrancar el estado en 0 y «corregirlo» al final** (sumar el valor inicial después del bucle). El resultado final puede salir bien, pero los valores intermedios y los eventos quedan mal.
2. **Filtrar casos que no cambian nada** (sumar 0 no altera un saldo) o, al revés, **no filtrar donde sí importa** (un movimiento de 0 puede crear una clave que no debería existir).
3. **Usar 0 como valor inicial de un extremo.** Funciona por accidente cuando el 0 hace de filtro y falla cuando todos los valores son positivos. El centinela correcto es `None`.
4. **Leer una clave que aún no existe** (`KeyError`) o usar `.get(clave)` sin valor por defecto (`None + 1` da `TypeError`).
5. **Escribir un `for` distinto para cada resultado** en vez de llevar varios acumuladores en un solo recorrido.
6. **No redondear el resultado que se entrega** (`477.5799999999999`) o comparar decimales con `==`.
7. **`min(lista_de_tuplas)` sin `key`:** compara por el primer elemento (la fecha), no por el monto.
8. **Un `continue` mal ubicado** que se salta un caso válido antes de que el chequeo importante ocurra.

# Módulo 1 — Diccionarios: agrupación y acumulación
⏱️ **50 min** · Objetivo: agrupar datos por categoría (total, conteo y más) con un solo recorrido, sin `KeyError` y entregando resultados redondeados.

## 1. Ciclo de vida de una clave

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 (print del arroz) → 
#   línea 2 (print de "leche" in ...) → 
#   línea 3 (¿qué se imprime al leer leche?) → 

inventario = {"arroz": 12, "azúcar": 8}

print(inventario["arroz"])              # línea 1
print("leche" in inventario)            # línea 2
try:
    print(inventario["leche"])          # línea 3
except KeyError as error:
    print("KeyError con la clave:", error)

**Teoría corta**

- Una clave **nace cuando le asignas un valor**: `d["leche"] = 5`.
- `d[clave]` solo **lee**. Si la clave no existe, Python lanza `KeyError` (y tu programa se detiene).
- `clave in d` pregunta **sin riesgo** si la clave ya existe y devuelve `True` o `False`.

Por eso, antes de leer una clave que quizá no exista, pregunta con `if clave in d`.

In [ ]:
inventario = {"arroz": 12, "azúcar": 8}

for producto in ["arroz", "leche"]:
    if producto in inventario:
        print(f"{producto}: hay {inventario[producto]} unidades")
    else:
        print(f"{producto}: todavía no existe como clave")

inventario["leche"] = 5          # asignar CREA la clave
print(inventario)

## 2. Patrón clásico: categoría → total y categoría → conteo

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   el diccionario final `totales` → 

totales = {}
for categoria, monto in [("lácteos", 8.5), ("bebidas", 4.0), ("lácteos", 6.5)]:
    if categoria in totales:
        totales[categoria] += monto
    else:
        totales[categoria] = monto
print(totales)

**Teoría corta**

El patrón tiene dos ramas:

- **`else` (la primera vez que aparece la clave):** se **inicializa** con el valor del elemento actual. Para un total es el `monto`; para un conteo es `1`.
- **`if` (las vueltas siguientes):** se **acumula** sobre lo que ya hay.

⚠️ Si en el `else` pones `= 0` para un total, **pierdes el primer monto de cada categoría**: el `0` reemplaza al valor de esa vuelta.

Mira la traza paso a paso (vuelta | elemento | estado | evento):

In [ ]:
ventas_demo = [
    ("2026-03-02", "lácteos", 8.50),
    ("2026-03-02", "abarrotes", 23.00),
    ("2026-03-03", "lácteos", 6.50),
    ("2026-03-03", "bebidas", 4.00),
    ("2026-03-04", "abarrotes", 15.00),
    ("2026-03-04", "lácteos", 9.00),
]

totales = {}
conteos = {}
vuelta = 0
print(f"{'vuelta':<7}| {'elemento':<36}| {'totales':<56}| evento")
for fecha, categoria, monto in ventas_demo:
    vuelta += 1
    if categoria in totales:
        totales[categoria] += monto
        conteos[categoria] += 1
        evento = f"'{categoria}' ya existía → se acumula"
    else:
        totales[categoria] = monto
        conteos[categoria] = 1
        evento = f"'{categoria}' es nueva → se inicializa"
    print(f"{vuelta:<7}| {str((fecha, categoria, monto)):<36}| {str(totales):<56}| {evento}")

print("\nconteos:", conteos)

## 3. `d.get(clave, 0)`: leer y acumular sin `if`

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 → 
#   línea 2 → 
#   línea 3 → 
#   línea 4 (¿qué pasa al sumar 50 a lo que devuelve .get sin valor por defecto?) → 

ventas_por_dia = {"lunes": 120.0}

print(ventas_por_dia.get("lunes", 0))       # línea 1
print(ventas_por_dia.get("martes", 0))      # línea 2
print(ventas_por_dia.get("martes"))         # línea 3
try:
    print(ventas_por_dia.get("martes") + 50)    # línea 4
except TypeError as error:
    print("TypeError:", error)

**Teoría corta**

- `d.get(clave, valor_por_defecto)` devuelve `d[clave]` si existe y, si no, el valor por defecto. **Nunca** lanza `KeyError`.
- Sin valor por defecto devuelve `None`, y `None + 50` da `TypeError`. Por eso, **al acumular, siempre pasa el valor por defecto** (`0` para totales y conteos).
- Acumular queda en una sola línea, sin `if/else`:

```python
totales[categoria] = totales.get(categoria, 0) + monto
```

- Con `.get` puedes llevar **varios acumuladores en un solo recorrido**: uno por diccionario (o por variable).

In [ ]:
ventas_demo = [
    ("2026-03-02", "lácteos", 8.50),
    ("2026-03-02", "abarrotes", 23.00),
    ("2026-03-03", "lácteos", 6.50),
    ("2026-03-03", "bebidas", 4.00),
    ("2026-03-04", "abarrotes", 15.00),
    ("2026-03-04", "lácteos", 9.00),
]

totales = {}
conteos = {}
vuelta = 0
print(f"{'vuelta':<7}| {'elemento':<36}| {'totales':<56}| conteos")
for fecha, categoria, monto in ventas_demo:
    vuelta += 1
    totales[categoria] = totales.get(categoria, 0) + monto   # total
    conteos[categoria] = conteos.get(categoria, 0) + 1       # conteo
    print(f"{vuelta:<7}| {str((fecha, categoria, monto)):<36}| {str(totales):<56}| {conteos}")

## 4. Recorrer un diccionario: `for k in d`, `.keys()`, `.values()`, `.items()`

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   qué imprime el primer for → 
#   keys() → 
#   values() → 
#   items() → 
#   el dict tras subir 10 % → 
#   qué pasa al agregar una clave dentro del for → 

precios = {"arroz": 4.5, "aceite": 9.9, "azúcar": 3.8}

for producto in precios:
    print(producto)
print(list(precios.keys()))
print(list(precios.values()))
print(list(precios.items()))

for producto in precios:                      # cambiar VALORES mientras recorres
    precios[producto] = round(precios[producto] * 1.1, 2)
print(precios)

copia = dict(precios)
try:
    for producto in copia:                    # agregar una CLAVE mientras recorres
        copia["leche"] = 5.0
except RuntimeError as error:
    print("RuntimeError:", error)

**Teoría corta**

| Forma | Recorres | Cada vuelta te da |
|---|---|---|
| `for k in d` / `d.keys()` | las claves | `k` |
| `d.values()` | los valores | `v` |
| `d.items()` | pares | `(k, v)` → `for k, v in d.items():` |

- **Cambiar valores** de claves existentes mientras recorres **está bien**.
- **Agregar o quitar claves** mientras recorres da `RuntimeError: dictionary changed size during iteration`. Si necesitas eso, recorre una copia (`list(d)`) o arma un diccionario nuevo.

In [ ]:
stock = {"arroz": 12, "aceite": 0, "azúcar": 7}

for producto, unidades in stock.items():
    estado = "agotado" if unidades == 0 else "disponible"
    print(f"{producto:<8} {unidades:>3}  {estado}")

print("unidades en total:", sum(stock.values()))

## 5. ¿Por qué sale `477.5799999999999`? Redondea al entregar

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 → 
#   línea 2 → 
#   línea 3 (total sin redondear) → 
#   línea 4 (total redondeado) → 

print(0.1 + 0.2)                            # línea 1
print(0.1 + 0.2 == 0.3)                     # línea 2

compras = [12.35, 8.20, 45.90, 3.15, 27.60, 9.85]
total = 0
for compra in compras:
    total += compra
print(total)                                # línea 3
print(round(total, 2))                      # línea 4

**Teoría corta**

- Python guarda los decimales en **binario**. Números como `0.1` no caben exactos, y al sumar muchos aparecen restos diminutos (`107.04999999999998`).
- **No es un error tuyo** ni de Python: es cómo funcionan los `float`.
- Regla: **acumula sin redondear y redondea al entregar** el resultado, con `round(x, 2)`. Redondear en cada paso no arregla nada y ensucia el cálculo.
- Recorrer un diccionario para redondear sus valores **al final** es válido (cambiar valores está permitido):

```python
for categoria in totales:
    totales[categoria] = round(totales[categoria], 2)
```

- Para **comparar** decimales no uses `==`: usa `round(x, 2) == y` o `math.isclose(x, y)`.

In [ ]:
import math

totales = {"lácteos": 0, "snacks": 0}
datos = [("lácteos", 12.80), ("snacks", 2.20), ("lácteos", 9.60), ("snacks", 1.10),
         ("lácteos", 15.45), ("snacks", 3.50)]

for categoria, monto in datos:
    totales[categoria] = totales.get(categoria, 0) + monto
print("crudo:     ", totales)

for categoria in totales:                       # al entregar: redondea
    totales[categoria] = round(totales[categoria], 2)
print("redondeado:", totales)

print(totales["snacks"] == 6.8, math.isclose(totales["snacks"], 6.8))

## 🏋️ Ejercicios del módulo 1
Recuerda: define primero tu predicción mental, escribe tu código y luego ejecuta la validación.

### 🟢 Básico Ejercicio 1.1 — Conteo por categoría

Una bodega registró sus ventas de tres días como `(fecha, categoria, monto)`. Cuenta **cuántas ventas hubo por categoría** con el patrón `if / else`.

**Variable a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `conteo_por_categoria` | `dict` | categoría (`str`) → cantidad de ventas (`int`) |

In [ ]:
ventas = [
    ("2026-05-04", "abarrotes", 23.40),
    ("2026-05-04", "bebidas", 7.50),
    ("2026-05-04", "lácteos", 12.80),
    ("2026-05-05", "abarrotes", 41.10),
    ("2026-05-05", "snacks", 3.50),
    ("2026-05-05", "bebidas", 5.00),
    ("2026-05-06", "abarrotes", 18.90),
    ("2026-05-06", "lácteos", 9.60),
    ("2026-05-06", "bebidas", 7.50),
    ("2026-05-06", "limpieza", 14.20),
]
_ventas_originales = list(ventas)   # copia para la validación (no la toques)

conteo_por_categoria = None   # dict: categoría → cantidad de ventas

# Tu código aquí

In [ ]:
assert isinstance(conteo_por_categoria, dict), "conteo_por_categoria debe ser un dict: ¿lo creaste con {}?"
assert set(conteo_por_categoria) == {"abarrotes", "bebidas", "lácteos", "snacks", "limpieza"}, \
    "Las categorías no coinciden: ¿falta alguna? La primera vez que aparece una clave hay que crearla."
assert sum(conteo_por_categoria.values()) == len(ventas), \
    "Cada venta debe contarse una sola vez: ¿inicializaste en 0 en el else y perdiste la primera venta de cada categoría?"
assert conteo_por_categoria == {"abarrotes": 3, "bebidas": 3, "lácteos": 2, "snacks": 1, "limpieza": 1}, \
    "Los conteos no coinciden: ¿estás sumando el monto en vez de sumar 1?"
assert ventas == _ventas_originales, "Cuidado: modificaste la lista original `ventas`."

print("✅ Ejercicio 1.1 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Necesitas un diccionario vacío y, por cada venta, decidir: ¿la categoría ya está en el diccionario? Si ya está, suma 1; si no, créala.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('Estructura: conteo_por_categoria = {} y luego for fecha, categoria, monto in ventas: if categoria in conteo_por_categoria: (suma 1) else: (empieza en 1).')

### 🟢 Básico Ejercicio 1.2 — Total por categoría con `.get`

Con las ventas de la semana `(fecha, categoria, monto)`, calcula el **total vendido por categoría** usando `.get(categoria, 0)` (sin `if / else`) y **entrega los totales redondeados a 2 decimales**.

**Variable a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `total_por_categoria` | `dict` | categoría (`str`) → total (`float`, redondeado a 2 decimales) |

In [ ]:
ventas_semana = [
    ("2026-05-11", "abarrotes", 24.90),
    ("2026-05-11", "lácteos", 12.80),
    ("2026-05-11", "snacks", 2.20),
    ("2026-05-12", "bebidas", 7.50),
    ("2026-05-12", "abarrotes", 18.35),
    ("2026-05-12", "lácteos", 9.60),
    ("2026-05-13", "limpieza", 14.20),
    ("2026-05-13", "abarrotes", 7.10),
    ("2026-05-13", "snacks", 1.10),
    ("2026-05-14", "lácteos", 15.45),
    ("2026-05-14", "bebidas", 5.00),
    ("2026-05-14", "snacks", 3.50),
    ("2026-05-14", "limpieza", 8.90),
    ("2026-05-15", "abarrotes", 31.60),
    ("2026-05-15", "bebidas", 6.30),
]
_ventas_originales = list(ventas_semana)   # copia para la validación (no la toques)

total_por_categoria = None   # dict: categoría → total (float, 2 decimales)

# Tu código aquí

In [ ]:
import math

esperado = {"abarrotes": 81.95, "lácteos": 37.85, "snacks": 6.80, "bebidas": 18.80, "limpieza": 23.10}

assert isinstance(total_por_categoria, dict), "total_por_categoria debe ser un dict: ¿lo creaste con {}?"
assert set(total_por_categoria) == set(esperado), "Las categorías no coinciden: ¿falta alguna?"
for categoria, total in esperado.items():
    assert math.isclose(total_por_categoria[categoria], total, abs_tol=0.005), \
        f"El total de '{categoria}' no coincide: ¿sumaste todos sus montos? ¿perdiste el primero al inicializar?"
assert all(round(valor, 2) == valor for valor in total_por_categoria.values()), \
    "¿Redondeaste a 2 decimales? Hay totales como 37.849999999999994: redondea con round(x, 2) al entregar."
assert ventas_semana == _ventas_originales, "Cuidado: modificaste la lista original `ventas_semana`."

print("✅ Ejercicio 1.2 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Usa un solo renglón dentro del for: total_por_categoria[categoria] = total_por_categoria.get(categoria, 0) + monto. El redondeo va aparte, cuando el for ya terminó.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('Después del for, recorre el diccionario: for categoria in total_por_categoria: total_por_categoria[categoria] = round(total_por_categoria[categoria], 2). Cambiar valores mientras recorres está permitido.')

### 🐛 Ejercicio 1.3 — Egresos por concepto (encuentra y arregla el bug)

Con los movimientos de una cuenta `(fecha, concepto, monto)`, se quiere el **gasto por concepto**: solo los **egresos** (montos negativos), **guardados en positivo** y redondeados a 2 decimales. Los ingresos y los montos `0` **no** deben aparecer.

El código de la celda siguiente **corre sin errores pero da resultados incorrectos**. Encuentra el error y arréglalo.

**Variable a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `gasto_por_concepto` | `dict` | concepto (`str`) → gasto (`float`, positivo, 2 decimales) |

> 🔎 Ojo: aquí **sí** conviene ignorar los montos 0. No por el cálculo (sumar 0 no cambia un total), sino porque un movimiento de 0 **crearía una clave** que no debería existir.

In [ ]:
movimientos = [
    ("2026-06-01", "sueldo", 1500.00),
    ("2026-06-02", "alquiler", -600.00),
    ("2026-06-03", "mercado", -85.40),
    ("2026-06-04", "ajuste", 0.0),
    ("2026-06-06", "yape", 120.00),
    ("2026-06-07", "mercado", -62.30),
    ("2026-06-09", "transporte", -18.50),
    ("2026-06-12", "mercado", -47.15),
    ("2026-06-14", "ajuste", 0.0),
    ("2026-06-15", "transporte", -21.00),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

# 🐛 Este código tiene un error típico. Arréglalo (puedes reescribirlo).
gasto_por_concepto = {}
for fecha, concepto, monto in movimientos:
    if monto != 0:
        gasto_por_concepto[concepto] = gasto_por_concepto.get(concepto, 0) + monto

In [ ]:
import math

esperado = {"alquiler": 600.00, "mercado": 194.85, "transporte": 39.50}

assert isinstance(gasto_por_concepto, dict), "gasto_por_concepto debe ser un dict."
assert "sueldo" not in gasto_por_concepto and "yape" not in gasto_por_concepto, \
    "¿Contaste ingresos como si fueran gastos? Solo cuentan los montos negativos."
assert "ajuste" not in gasto_por_concepto, \
    "Un concepto con solo montos 0 no debe aparecer: ¿filtras con `monto != 0` en vez de `monto < 0`?"
assert all(valor > 0 for valor in gasto_por_concepto.values()), \
    "¿Dejaste los egresos en negativo? Guárdalos en positivo (usa -monto)."
assert set(gasto_por_concepto) == set(esperado), "Los conceptos no coinciden con los egresos reales."
for concepto, gasto in esperado.items():
    assert math.isclose(gasto_por_concepto[concepto], gasto, abs_tol=0.005), \
        f"El gasto de '{concepto}' no coincide: revisa qué sumas en cada vuelta."
assert all(round(valor, 2) == valor for valor in gasto_por_concepto.values()), \
    "¿Redondeaste a 2 decimales al entregar?"
assert movimientos == _movimientos_originales, "Cuidado: modificaste la lista original `movimientos`."

print("✅ Ejercicio 1.3 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Hay dos fallos en el mismo if/acumulación: uno decide QUÉ movimientos entran (mira quién pasa el filtro) y otro decide QUÉ SE SUMA (mira el signo). Además falta entregar redondeado.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('El filtro correcto para «solo egresos» es monto < 0: excluye ingresos (positivos) y ajustes (0) a la vez. Al acumular, suma -monto para guardar el gasto en positivo. Redondea con un for sobre el diccionario al final.')

### 🔴 Integrador Ejercicio 1.4 — Tres resultados en un solo recorrido

Un minimarket anotó sus compras a proveedores del mes como `(fecha, categoria, monto)` (todos los montos son gastos positivos). Calcula, **recorriendo la lista `gastos` una sola vez** para los totales y los conteos:

**Variables a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `total_por_categoria` | `dict` | categoría → total gastado (`float`, 2 decimales) |
| `cantidad_por_categoria` | `dict` | categoría → número de compras (`int`) |
| `promedio_por_categoria` | `dict` | categoría → total / cantidad (`float`, **redondeado** a 2 decimales) |
| `categoria_mayor_gasto` | `str` | categoría con mayor total, **buscada a mano** recorriendo `.items()` (sin `max`) |

Pistas de método: acumula total y cantidad en el mismo `for`; redondea y calcula el promedio **al entregar**, con el diccionario ya armado.

In [ ]:
gastos = [
    ("2026-07-01", "abarrotes", 342.50),
    ("2026-07-01", "limpieza", 58.90),
    ("2026-07-03", "bebidas", 126.40),
    ("2026-07-04", "abarrotes", 215.75),
    ("2026-07-06", "lácteos", 89.30),
    ("2026-07-08", "limpieza", 44.10),
    ("2026-07-09", "bebidas", 98.60),
    ("2026-07-11", "abarrotes", 187.20),
    ("2026-07-12", "lácteos", 76.90),
    ("2026-07-14", "snacks", 63.15),
    ("2026-07-15", "bebidas", 110.00),
    ("2026-07-15", "limpieza", 39.95),
]
_gastos_originales = list(gastos)   # copia para la validación (no la toques)

total_por_categoria = None       # dict: categoría → total (float, 2 decimales)
cantidad_por_categoria = None    # dict: categoría → cantidad de compras (int)
promedio_por_categoria = None    # dict: categoría → promedio (float, 2 decimales)
categoria_mayor_gasto = None     # str: categoría con mayor total

# Tu código aquí

In [ ]:
import math

tot = {"abarrotes": 745.45, "limpieza": 142.95, "bebidas": 335.00, "lácteos": 166.20, "snacks": 63.15}
cant = {"abarrotes": 3, "limpieza": 3, "bebidas": 3, "lácteos": 2, "snacks": 1}
prom = {"abarrotes": 248.48, "limpieza": 47.65, "bebidas": 111.67, "lácteos": 83.10, "snacks": 63.15}

assert isinstance(total_por_categoria, dict) and isinstance(cantidad_por_categoria, dict) \
    and isinstance(promedio_por_categoria, dict), "Los tres resultados deben ser diccionarios."
assert set(total_por_categoria) == set(tot), "total_por_categoria: las categorías no coinciden."
for categoria, valor in tot.items():
    assert math.isclose(total_por_categoria[categoria], valor, abs_tol=0.005), \
        f"Total de '{categoria}' incorrecto: ¿perdiste la primera compra al inicializar?"
assert all(round(v, 2) == v for v in total_por_categoria.values()), \
    "¿Redondeaste los totales a 2 decimales?"
assert cantidad_por_categoria == cant, "cantidad_por_categoria incorrecta: cuenta 1 por compra, no el monto."
assert set(promedio_por_categoria) == set(prom), "promedio_por_categoria: las categorías no coinciden."
for categoria, valor in prom.items():
    assert math.isclose(promedio_por_categoria[categoria], valor, abs_tol=0.005), \
        f"Promedio de '{categoria}' incorrecto: total / cantidad, redondeado a 2 decimales."
assert all(round(v, 2) == v for v in promedio_por_categoria.values()), \
    "¿Redondeaste los promedios a 2 decimales?"
assert categoria_mayor_gasto == "abarrotes", \
    "categoria_mayor_gasto incorrecta: compara TOTALES por categoría, no montos individuales."
assert gastos == _gastos_originales, "Cuidado: modificaste la lista original `gastos`."

print("✅ Ejercicio 1.4 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Un solo for sobre gastos con dos acumuladores: total_por_categoria[categoria] = total_por_categoria.get(categoria, 0) + monto y lo mismo para cantidad_por_categoria (sumando 1).')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('Después del for, recorre total_por_categoria.items(): calcula el promedio, redondea el total y busca el mayor con una variable que empiece en None (mayor_total = None; if mayor_total is None or total > mayor_total: ...). Cuidado: si redondeas el total dentro de ese recorrido, cambia el valor, no la clave.')

## 🔒 Soluciones del módulo 1
Abre cada celda **solo después de intentarlo**.

In [ ]:
# @title 🔒 Solución 1.1 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Inicializar el conteo en 0 dentro del else: la primera venta de cada categoría no se cuenta.
# Leer conteo_por_categoria[categoria] antes de crearla provoca KeyError.
# Sumar el monto en lugar de 1 convierte el conteo en un total.

ventas = [
    ("2026-05-04", "abarrotes", 23.40),
    ("2026-05-04", "bebidas", 7.50),
    ("2026-05-04", "lácteos", 12.80),
    ("2026-05-05", "abarrotes", 41.10),
    ("2026-05-05", "snacks", 3.50),
    ("2026-05-05", "bebidas", 5.00),
    ("2026-05-06", "abarrotes", 18.90),
    ("2026-05-06", "lácteos", 9.60),
    ("2026-05-06", "bebidas", 7.50),
    ("2026-05-06", "limpieza", 14.20),
]
_ventas_originales = list(ventas)   # copia para la validación (no la toques)

conteo_por_categoria = {}
for fecha, categoria, monto in ventas:
    if categoria in conteo_por_categoria:
        conteo_por_categoria[categoria] += 1      # vueltas siguientes: acumular
    else:
        conteo_por_categoria[categoria] = 1       # primera vez: inicializar en 1

In [ ]:
# @title 🔒 Solución 1.2 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Usar .get(categoria) sin el 0 devuelve None y None + monto da TypeError.
# Entregar el total sin redondear (37.849999999999994): es normal en float, se redondea al final.
# Recorrer las ventas otra vez para redondear: basta recorrer el diccionario ya armado.

ventas_semana = [
    ("2026-05-11", "abarrotes", 24.90),
    ("2026-05-11", "lácteos", 12.80),
    ("2026-05-11", "snacks", 2.20),
    ("2026-05-12", "bebidas", 7.50),
    ("2026-05-12", "abarrotes", 18.35),
    ("2026-05-12", "lácteos", 9.60),
    ("2026-05-13", "limpieza", 14.20),
    ("2026-05-13", "abarrotes", 7.10),
    ("2026-05-13", "snacks", 1.10),
    ("2026-05-14", "lácteos", 15.45),
    ("2026-05-14", "bebidas", 5.00),
    ("2026-05-14", "snacks", 3.50),
    ("2026-05-14", "limpieza", 8.90),
    ("2026-05-15", "abarrotes", 31.60),
    ("2026-05-15", "bebidas", 6.30),
]
_ventas_originales = list(ventas_semana)   # copia para la validación (no la toques)

total_por_categoria = {}
for fecha, categoria, monto in ventas_semana:
    total_por_categoria[categoria] = total_por_categoria.get(categoria, 0) + monto

for categoria in total_por_categoria:          # al entregar: redondear a 2 decimales
    total_por_categoria[categoria] = round(total_por_categoria[categoria], 2)

In [ ]:
# @title 🔒 Solución 1.3 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# `monto != 0` deja pasar los ingresos: se mezclan con los gastos y además llegan con signo positivo/negativo mezclado.
# Con `monto < 0` se descartan ingresos y ceros de una vez; al sumar -monto el gasto queda positivo.
# Aquí ignorar el 0 importa porque evita crear la clave 'ajuste'; al calcular un saldo, en cambio, filtrar el 0 es innecesario.

movimientos = [
    ("2026-06-01", "sueldo", 1500.00),
    ("2026-06-02", "alquiler", -600.00),
    ("2026-06-03", "mercado", -85.40),
    ("2026-06-04", "ajuste", 0.0),
    ("2026-06-06", "yape", 120.00),
    ("2026-06-07", "mercado", -62.30),
    ("2026-06-09", "transporte", -18.50),
    ("2026-06-12", "mercado", -47.15),
    ("2026-06-14", "ajuste", 0.0),
    ("2026-06-15", "transporte", -21.00),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

gasto_por_concepto = {}
for fecha, concepto, monto in movimientos:
    if monto < 0:                                            # solo egresos (excluye ingresos y 0)
        gasto_por_concepto[concepto] = gasto_por_concepto.get(concepto, 0) - monto   # en positivo

for concepto in gasto_por_concepto:                          # entregar redondeado
    gasto_por_concepto[concepto] = round(gasto_por_concepto[concepto], 2)

In [ ]:
# @title 🔒 Solución 1.4 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Promediar con el total ya redondeado o redondear antes de dividir cambia el resultado: calcula total / cantidad y redondea al entregar.
# Buscar el máximo comparando montos sueltos en vez de totales por categoría.
# Empezar mayor_total en 0 funciona aquí (todo es positivo) pero es el hábito que falla en el módulo 2: usa None.

gastos = [
    ("2026-07-01", "abarrotes", 342.50),
    ("2026-07-01", "limpieza", 58.90),
    ("2026-07-03", "bebidas", 126.40),
    ("2026-07-04", "abarrotes", 215.75),
    ("2026-07-06", "lácteos", 89.30),
    ("2026-07-08", "limpieza", 44.10),
    ("2026-07-09", "bebidas", 98.60),
    ("2026-07-11", "abarrotes", 187.20),
    ("2026-07-12", "lácteos", 76.90),
    ("2026-07-14", "snacks", 63.15),
    ("2026-07-15", "bebidas", 110.00),
    ("2026-07-15", "limpieza", 39.95),
]
_gastos_originales = list(gastos)   # copia para la validación (no la toques)

total_por_categoria = {}
cantidad_por_categoria = {}
for fecha, categoria, monto in gastos:                       # un solo recorrido, dos acumuladores
    total_por_categoria[categoria] = total_por_categoria.get(categoria, 0) + monto
    cantidad_por_categoria[categoria] = cantidad_por_categoria.get(categoria, 0) + 1

promedio_por_categoria = {}
categoria_mayor_gasto = None
mayor_total = None                                           # centinela: aún no hay mejor
for categoria, total in total_por_categoria.items():
    promedio_por_categoria[categoria] = round(total / cantidad_por_categoria[categoria], 2)
    if mayor_total is None or total > mayor_total:
        mayor_total = total
        categoria_mayor_gasto = categoria

for categoria in total_por_categoria:                        # entregar redondeado
    total_por_categoria[categoria] = round(total_por_categoria[categoria], 2)

# Módulo 2 — Estado acumulativo y control de flujo
⏱️ **70 min** · Objetivo: llevar un **estado** (saldo, stock) que cambia en cada vuelta, detectar **eventos** mientras avanza y buscar extremos sin trampas.

## 1. Agregación final (`sum`) vs seguimiento paso a paso

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 (saldo_inicial + sum) → 
#   línea 2 (la lista de saldos) → 
#   línea 3 (el saldo más bajo) → 

saldo_inicial = 60.0
montos = [45.0, -120.0, 10.0, 0.0, 40.0, -40.0, 55.0, -25.0]

print(saldo_inicial + sum(montos))          # línea 1

saldos = []
saldo = saldo_inicial
for monto in montos:
    saldo += monto
    saldos.append(saldo)
print(saldos)                                # línea 2
print(min(saldos))                           # línea 3

**Teoría corta**

- **Agregación final** (`sum`, `len`, `min`…): un solo número al final. Responde «¿cuánto quedó?».
- **Seguimiento paso a paso:** guardas el estado en cada vuelta (una lista de saldos). Responde «¿**cuándo** pasó?», «¿**cuántos** días?», «¿**cuál fue el peor** momento?».
- Con los mismos datos, `sum` te da el saldo final, pero **no** te dice si en el camino la cuenta quedó en rojo.

Traza (vuelta | elemento | estado | evento):

In [ ]:
saldo_inicial_demo = 60.0
movimientos_demo = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]

saldo = saldo_inicial_demo
saldos = []
print(f"saldo inicial: {saldo:.2f}")
print(f"{'vuelta':<7}| {'fecha':<11}| {'monto':>8} | {'saldo':>8} | evento")
vuelta = 0
for fecha, concepto, monto in movimientos_demo:
    vuelta += 1
    saldo += monto
    saldos.append(saldo)
    evento = "en rojo" if saldo < 0 else ""
    print(f"{vuelta:<7}| {fecha:<11}| {monto:>8.2f} | {saldo:>8.2f} | {evento}")

print()
print("saldo final con sum:", saldo_inicial_demo + sum(m[2] for m in movimientos_demo))
print("saldo final paso a paso:", saldos[-1], "| saldo más bajo:", min(saldos))

## 2. El estado arranca en el valor inicial, no en 0

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   vuelta 1: saldo_mal saldo_bien → 
#   vuelta 2: saldo_mal saldo_bien → 
#   vuelta 3: saldo_mal saldo_bien → 
#   última línea (True o False) → 

saldo_inicial = 60.0
montos = [45.0, -120.0, 10.0]

saldo_mal = 0                  # arrancar en 0...
saldo_bien = saldo_inicial     # ...vs. arrancar en el valor inicial
for monto in montos:
    saldo_mal += monto
    saldo_bien += monto
    print(saldo_mal, saldo_bien)

print(saldo_mal + saldo_inicial == saldo_bien)

**Teoría corta**

Arrancar en 0 y sumar `saldo_inicial` **al final** da el mismo resultado final, y por eso parece correcto. Pero los **valores intermedios están mal**: cada saldo del camino está desplazado por `saldo_inicial`, así que cualquier pregunta sobre el camino (¿estuvo en rojo?, ¿cuál fue el mínimo?) da respuestas falsas.

**Regla:** el estado nace con el valor que tenía **antes** del primer elemento.

In [ ]:
saldo_inicial_demo = 60.0
movimientos_demo = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]

saldo_mal = 0
saldo_bien = saldo_inicial_demo
vuelta = 0
print(f"{'vuelta':<7}| {'monto':>8} | {'saldo_mal':>9} | {'saldo_bien':>10} | ¿rojo según mal? | ¿rojo según bien?")
for fecha, concepto, monto in movimientos_demo:
    vuelta += 1
    saldo_mal += monto
    saldo_bien += monto
    print(f"{vuelta:<7}| {monto:>8.2f} | {saldo_mal:>9.2f} | {saldo_bien:>10.2f} | {str(saldo_mal < 0):<16} | {saldo_bien < 0}")

## 3. Eventos dentro del bucle: primero, todos y cruces

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   dias_en_rojo → 
#   entradas (veces que la cuenta ENTRA en rojo) → 

saldo = 60.0
saldo_anterior = saldo
dias_en_rojo = 0
entradas = 0
for monto in [45.0, -120.0, 10.0, 0.0, 40.0, -40.0, 55.0, -25.0]:
    saldo += monto
    if saldo < 0:
        dias_en_rojo += 1
        if saldo_anterior >= 0:
            entradas += 1
    saldo_anterior = saldo
print(dias_en_rojo, entradas)

**Teoría corta**

Tres preguntas distintas sobre el mismo recorrido:

| Pregunta | Patrón |
|---|---|
| **Primer** día en sobregiro | guarda la fecha y `break` (deja `None` si nunca ocurre) |
| **Todos** los días en sobregiro | `lista.append(fecha)` en cada vuelta con `saldo < 0` |
| **Cruces** de ≥ 0 a < 0 | necesitas recordar el `saldo_anterior` |

Diferencia clave:
- **Estar** en sobregiro es un *estado*: `saldo < 0` (basta el saldo actual).
- **Entrar** en sobregiro es un *evento*: `saldo_anterior >= 0 and saldo < 0` (necesitas el pasado).

⚠️ Actualiza `saldo_anterior = saldo` **al final** de cada vuelta, después de haberlo usado.

In [ ]:
saldo_inicial_demo = 60.0
movimientos_demo = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]

# a) PRIMER día en sobregiro: break y None si nunca ocurre
primer_dia = None
saldo = saldo_inicial_demo
for fecha, concepto, monto in movimientos_demo:
    saldo += monto
    if saldo < 0:
        primer_dia = fecha
        break
print("primer día en sobregiro:", primer_dia)

# b) TODOS los días en sobregiro
dias_en_rojo = []
saldo = saldo_inicial_demo
for fecha, concepto, monto in movimientos_demo:
    saldo += monto
    if saldo < 0:
        dias_en_rojo.append(fecha)
print("días en sobregiro:", dias_en_rojo)

# c) CRUCES (entradas y salidas) con traza
saldo = saldo_inicial_demo
saldo_anterior = saldo
entradas = 0
vuelta = 0
print()
print(f"{'vuelta':<7}| {'fecha':<11}| {'monto':>8} | {'anterior → nuevo':<19}| {'en rojo':<8}| evento")
for fecha, concepto, monto in movimientos_demo:
    vuelta += 1
    saldo += monto
    if saldo_anterior >= 0 and saldo < 0:
        evento = "ENTRA en sobregiro"
        entradas += 1
    elif saldo_anterior < 0 and saldo >= 0:
        evento = "SALE del sobregiro"
    elif saldo < 0:
        evento = "sigue en sobregiro"
    else:
        evento = "-"
    print(f"{vuelta:<7}| {fecha:<11}| {monto:>8.2f} | {saldo_anterior:>7.2f} → {saldo:>7.2f} | {str(saldo < 0):<8}| {evento}")
    saldo_anterior = saldo          # se actualiza AL FINAL de la vuelta
print("\nentradas en sobregiro:", entradas)

**Gráfico ilustrativo** (aquí sí usamos `matplotlib`): el saldo en el tiempo, la línea del 0 y los días en sobregiro marcados.

In [ ]:
saldo_inicial_demo = 60.0
movimientos_demo = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]

import matplotlib.pyplot as plt

dias, saldos = [], []
dias_rojos, saldos_rojos = [], []
saldo = saldo_inicial_demo
for fecha, concepto, monto in movimientos_demo:
    saldo += monto
    dias.append(fecha[5:])                  # "02-01"
    saldos.append(saldo)
    if saldo < 0:
        dias_rojos.append(fecha[5:])
        saldos_rojos.append(saldo)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.axhspan(min(saldos) - 15, 0, color="#c0392b", alpha=0.07)          # zona en rojo
ax.plot(dias, saldos, color="#2a6fbb", linewidth=2, marker="o", markersize=6,
        label="Saldo al cierre del día")
ax.scatter(dias_rojos, saldos_rojos, color="#c0392b", s=90, zorder=3,
           label="Día en sobregiro")
ax.axhline(0, color="#444444", linewidth=1.2)                          # línea del 0
ax.set_title("Saldo de la cuenta día a día (S/)")
ax.set_xlabel("Día (mes-día)")
ax.set_ylabel("Saldo (S/)")
ax.grid(axis="y", color="#dddddd", linewidth=0.8)
ax.legend(frameon=False, loc="upper right")
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.show()

## 4. Extremos a mano: `None` como centinela y el registro completo

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 (menor venta con menor = 0) → 
#   línea 2 (menor venta con None) → 
#   línea 3 (fecha con `<` en el empate) → 
#   línea 4 (fecha con `<=` en el empate) → 

ventas_dia = [("2026-04-01", 38.5), ("2026-04-02", 52.0), ("2026-04-03", 29.9), ("2026-04-04", 61.4)]

menor = 0
for fecha, venta in ventas_dia:
    if venta < menor:
        menor = venta
print(menor)                                   # línea 1

menor = None
for fecha, venta in ventas_dia:
    if menor is None or venta < menor:
        menor = venta
print(menor)                                   # línea 2

empate = [("2026-04-01", 30.0), ("2026-04-02", 25.0), ("2026-04-03", 25.0)]
mejor_valor, mejor_registro = None, None
for registro in empate:
    if mejor_valor is None or registro[1] < mejor_valor:
        mejor_valor, mejor_registro = registro[1], registro
print(mejor_registro[0])                       # línea 3

mejor_valor, mejor_registro = None, None
for registro in empate:
    if mejor_valor is None or registro[1] <= mejor_valor:
        mejor_valor, mejor_registro = registro[1], registro
print(mejor_registro[0])                       # línea 4

**Teoría corta**

- Un **centinela** es un valor especial que significa «todavía no hay ninguno». `None` es el centinela correcto porque **no puede confundirse con un dato real**.
- Patrón: `mejor_valor = None`, `mejor_registro = None`; en el `if`, pregunta primero `mejor_valor is None or ...` (con `is`, no `==`).
- **Guarda el registro completo** (`mejor_registro`), no solo el valor: así recuperas fecha y concepto sin recorrer otra vez.
- **Inicializar en 0 es un bug latente.** Con `monto_mas_negativo = 0` funciona cuando hay egresos, porque el 0 hace de filtro. Pero si todos los datos son positivos (la venta más pequeña) el 0 «gana» y nunca se actualiza.
- **Empates:** con `<` te quedas con el **primero** que alcanzó el extremo; con `<=`, con el **último**.

In [ ]:
conjuntos = {
    "movimientos mixtos (hay egresos)": [("2026-04-01", 120.0), ("2026-04-02", -35.0), ("2026-04-03", -80.0)],
    "solo ventas (todo positivo)": [("2026-04-01", 38.5), ("2026-04-02", 52.0), ("2026-04-03", 29.9)],
}

for nombre, registros in conjuntos.items():
    con_cero = 0
    con_none = None
    registro_none = None
    for registro in registros:
        if registro[1] < con_cero:
            con_cero = registro[1]
        if con_none is None or registro[1] < con_none:
            con_none = registro[1]
            registro_none = registro
    print(f"{nombre}\n   inicializando en 0 → {con_cero}\n   inicializando en None → {registro_none}\n")

## 5. `break` vs `continue` (y una trampa)

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   dias_en_rojo con el continue mal puesto (¿4 o 3?) → 
#   dias_en_rojo sin el continue → 

movimientos = [("2026-02-01", "ventas", 45.0), ("2026-02-02", "proveedor", -120.0),
               ("2026-02-03", "ventas", 10.0), ("2026-02-04", "ajuste", 0.0),
               ("2026-02-05", "ventas", 40.0), ("2026-02-06", "luz", -40.0)]

saldo = 60.0
dias_en_rojo = 0
for fecha, concepto, monto in movimientos:
    if monto == 0:
        continue                 # «sumar 0 no cambia nada, lo salto»
    saldo += monto
    if saldo < 0:
        dias_en_rojo += 1
print(dias_en_rojo)              # con el continue

saldo = 60.0
dias_en_rojo = 0
for fecha, concepto, monto in movimientos:
    saldo += monto
    if saldo < 0:
        dias_en_rojo += 1
print(dias_en_rojo)              # sin el continue

**Teoría corta**

- **`break`:** termina el bucle completo. Úsalo cuando ya tienes la respuesta (el primer día en rojo).
- **`continue`:** salta al **siguiente elemento**; todo lo que hay **debajo** del `continue`, en esa vuelta, **no se ejecuta**.
- **La trampa:** un `continue` para saltar montos 0, puesto **antes** del chequeo de sobregiro, se salta un día que **sí estaba** en sobregiro (el saldo sigue negativo aunque el monto sea 0). El día existe aunque el monto no cambie nada.
- Regla práctica: antes de escribir un `continue`, pregunta «¿todo lo que viene después es irrelevante para este elemento?». Con saldos, casi nunca lo es: **sumar 0 no cambia el saldo, así que no hace falta filtrarlo**.

In [ ]:
saldo_inicial_demo = 60.0
movimientos_demo = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]

# Misma cuenta, dos versiones: el continue mal ubicado se salta el 2026-02-04
for version in ("con continue antes del chequeo", "sin continue (correcta)"):
    saldo = saldo_inicial_demo
    dias_en_rojo = []
    print(version)
    for fecha, concepto, monto in movimientos_demo:
        if version.startswith("con") and monto == 0:
            print(f"   {fecha}  monto 0 → continue: se salta el chequeo")
            continue
        saldo += monto
        if saldo < 0:
            dias_en_rojo.append(fecha)
        print(f"   {fecha}  saldo {saldo:>7.2f}  {'en rojo' if saldo < 0 else ''}")
    print(f"   → días en sobregiro: {len(dias_en_rojo)}\n")

## 🏋️ Ejercicios del módulo 2

### 🟢 Básico Ejercicio 2.1 — Saldos paso a paso

Una bodega abre junio con `saldo_inicial = 250.0`. Con sus movimientos `(fecha, concepto, monto)`, calcula el **saldo después de cada movimiento** y el saldo final.

**Variables a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `saldos` | `list` de `float` | saldo al cierre de **cada** movimiento, redondeado a 2 decimales |
| `saldo_final` | `float` | último saldo, redondeado a 2 decimales |

In [ ]:
saldo_inicial = 250.0
movimientos = [
    ("2026-06-01", "proveedor", -180.50),
    ("2026-06-02", "ventas del día", 95.20),
    ("2026-06-03", "ajuste", 0.0),
    ("2026-06-04", "luz", -64.30),
    ("2026-06-05", "ventas del día", 132.75),
    ("2026-06-06", "agua", -28.10),
    ("2026-06-07", "proveedor", -90.00),
    ("2026-06-08", "ventas del día", 210.40),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

saldos = None        # list de float (2 decimales), un saldo por movimiento
saldo_final = None   # float (2 decimales)

# Tu código aquí

In [ ]:
assert isinstance(saldos, list), "saldos debe ser una lista: ¿la creaste con [] y usaste append?"
assert len(saldos) == len(movimientos), \
    "Debe haber un saldo por cada movimiento: ¿filtraste los montos 0? Sumar 0 no cambia el saldo, pero ese día existe."
assert saldos[0] == 69.5, \
    "El primer saldo debe ser saldo_inicial + primer monto (250 - 180.50): ¿arrancaste el saldo en 0?"
assert all(round(s, 2) == s for s in saldos), "¿Redondeaste a 2 decimales cada saldo que guardas?"
assert saldos == [69.5, 164.7, 164.7, 100.4, 233.15, 205.05, 115.05, 325.45], \
    "Los saldos no coinciden: revisa que el estado acumule (saldo += monto) y no se reinicie."
assert saldo_final == 325.45, "saldo_final debe ser 325.45 (redondeado a 2 decimales)."
assert movimientos == _movimientos_originales, "Cuidado: modificaste la lista original `movimientos`."

print("✅ Ejercicio 2.1 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Necesitas una variable de estado (saldo) y una lista vacía (saldos). En cada vuelta: actualiza el estado y guarda una copia del estado en la lista.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('saldo = saldo_inicial (¡no 0!), saldos = []; for fecha, concepto, monto in movimientos: saldo += monto; saldos.append(round(saldo, 2)). Al final, saldo_final sale del último saldo.')

### 🟡 Intermedio Ejercicio 2.2 — Primer día de sobregiro

Halla la **fecha del primer día** en que el saldo queda **por debajo de 0**. Usa `break` y `None` como resultado cuando **nunca** ocurre. Se prueba con dos cuentas: una que sí entra en sobregiro (`a`) y una que nunca lo hace (`b`).

**Variables a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `primer_dia_a` | `str` o `None` | fecha ISO del primer saldo < 0 en `movimientos_a` |
| `primer_dia_b` | `str` o `None` | igual, para `movimientos_b` (debe quedar `None` si nunca ocurre) |

Escribe el mismo recorrido dos veces (una por cuenta): en otra sesión verás cómo evitar la repetición.

In [ ]:
saldo_inicial_a = 120.0
movimientos_a = [
    ("2026-07-01", "proveedor", -75.00),
    ("2026-07-02", "ventas", 60.00),
    ("2026-07-03", "luz", -50.00),
    ("2026-07-04", "ajuste", 0.00),
    ("2026-07-05", "proveedor", -98.50),
    ("2026-07-06", "ventas", 40.00),
    ("2026-07-07", "ventas", 70.00),
]

saldo_inicial_b = 500.0
movimientos_b = [
    ("2026-07-01", "proveedor", -120.00),
    ("2026-07-02", "ventas", 85.50),
    ("2026-07-03", "agua", -31.20),
    ("2026-07-04", "luz", -58.90),
    ("2026-07-05", "ventas", 60.00),
    ("2026-07-06", "proveedor", -140.25),
]
_originales = (list(movimientos_a), list(movimientos_b))   # copias para la validación (no las toques)

primer_dia_a = "sin resolver"   # str o None
primer_dia_b = "sin resolver"   # str o None

# Tu código aquí

In [ ]:
assert primer_dia_a == "2026-07-05", \
    "primer_dia_a incorrecto. Si obtienes '2026-07-06' te falta el break; si arrancaste el saldo en 0 el sobregiro aparece demasiado pronto."
assert primer_dia_b is None, \
    "primer_dia_b debe ser None: en la cuenta b nunca hay sobregiro. ¿Devolviste 0, '' o la última fecha?"
assert (movimientos_a, movimientos_b) == _originales, "Cuidado: modificaste una de las listas originales."

print("✅ Ejercicio 2.2 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Parte de primer_dia = None (aún no hay). Recorre con el saldo arrancando en saldo_inicial; en el momento en que saldo < 0, guarda la fecha y sal del bucle.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('saldo = saldo_inicial_a; primer_dia_a = None; for fecha, concepto, monto in movimientos_a: saldo += monto; if saldo < 0: primer_dia_a = fecha; break. Repite con la cuenta b, empezando de nuevo saldo y primer_dia_b.')

### 🐛 Ejercicio 2.3 — Saldo mínimo y su fecha (encuentra y arregla el bug)

La cuenta abre agosto con `saldo_inicial = 400.0` y **nunca baja de 0**. Se quiere el **saldo más bajo alcanzado** después de un movimiento y la **fecha** en que ocurrió (si hay empate, la primera).

El código siguiente **corre sin errores pero da un resultado incorrecto**. Encuentra el error y arréglalo.

**Variables a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `saldo_minimo` | `float` | menor saldo tras un movimiento (2 decimales) |
| `fecha_saldo_minimo` | `str` | fecha ISO de ese saldo (la primera si hay empate) |

In [ ]:
saldo_inicial = 400.0
movimientos = [
    ("2026-08-01", "proveedor", -150.00),
    ("2026-08-02", "ventas", 80.00),
    ("2026-08-03", "luz", -95.50),
    ("2026-08-04", "ventas", 120.30),
    ("2026-08-05", "alquiler", -200.00),
    ("2026-08-06", "ajuste", 0.00),
    ("2026-08-07", "ventas", 99.00),
    ("2026-08-08", "agua", -44.10),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

# 🐛 Este código tiene un error típico. Arréglalo (puedes reescribirlo).
saldo = saldo_inicial
saldo_minimo = 0
fecha_saldo_minimo = None
for fecha, concepto, monto in movimientos:
    saldo += monto
    if saldo < saldo_minimo:
        saldo_minimo = saldo
        fecha_saldo_minimo = fecha

In [ ]:
assert fecha_saldo_minimo is not None, \
    "fecha_saldo_minimo quedó en None: ¿inicializaste saldo_minimo en 0? Aquí ningún saldo baja de 0, así que nunca se actualiza."
assert saldo_minimo != 0, \
    "saldo_minimo quedó en 0: el 0 «gana» porque todos los saldos son positivos. Usa None como centinela."
assert round(saldo_minimo, 2) == 154.8, "saldo_minimo incorrecto: debe ser el menor saldo tras un movimiento (154.8)."
assert fecha_saldo_minimo == "2026-08-05", \
    "La fecha es la del PRIMER día con ese saldo mínimo: ¿usaste <= y te quedaste con el último empate?"
assert round(saldo_minimo, 2) == saldo_minimo, "¿Redondeaste saldo_minimo a 2 decimales?"
assert movimientos == _movimientos_originales, "Cuidado: modificaste la lista original `movimientos`."

print("✅ Ejercicio 2.3 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Pregúntate qué pasa con el valor inicial del mínimo cuando NINGÚN saldo es menor que él. ¿Quién es el «mejor hasta ahora» antes de mirar el primer movimiento?')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('Cambia saldo_minimo = 0 por None y ajusta la condición del if para que el primer saldo siempre se acepte: if saldo_minimo is None or saldo < saldo_minimo. Redondea al final.')

### 🔴 Integrador Ejercicio 2.4 — Kárdex de un producto

Una bodega lleva el kárdex del aceite de 1 L: `stock_inicial = 40` unidades y `stock_minimo = 15`. Cada movimiento es `(fecha, tipo, cantidad)` con `tipo` igual a `"entrada"` (suma) o `"salida"` (resta). **En un solo recorrido**, obtén:

**Variables a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `stock_final` | `int` | stock tras el último movimiento |
| `fechas_bajo_minimo` | `list` de `str` | fecha de cada movimiento tras el cual el stock queda **< `stock_minimo`** |
| `fecha_recuperacion` | `str` o `None` | **primera** fecha en que el stock pasa de estar bajo el mínimo a **> `stock_minimo`** |
| `mayor_salida` | `tuple` | el movimiento **completo** de mayor cantidad entre las **salidas** (en empate, el primero) |

In [ ]:
stock_inicial = 40
stock_minimo = 15
movimientos_kardex = [
    ("2026-09-01", "salida", 12),
    ("2026-09-02", "salida", 9),
    ("2026-09-03", "salida", 7),
    ("2026-09-04", "salida", 4),
    ("2026-09-05", "entrada", 20),
    ("2026-09-06", "salida", 15),
    ("2026-09-07", "salida", 3),
    ("2026-09-08", "entrada", 30),
    ("2026-09-09", "salida", 15),
    ("2026-09-10", "salida", 15),
]
_kardex_original = list(movimientos_kardex)   # copia para la validación (no la toques)

stock_final = None           # int
fechas_bajo_minimo = None    # list de str
fecha_recuperacion = None    # str o None
mayor_salida = None          # tuple (fecha, tipo, cantidad)

# Tu código aquí

In [ ]:
assert stock_final == 10, "stock_final debe ser 10: ¿las entradas suman y las salidas restan?"
assert fechas_bajo_minimo == ["2026-09-03", "2026-09-04", "2026-09-06", "2026-09-07", "2026-09-10"], \
    "fechas_bajo_minimo incorrecta: es el stock DESPUÉS de cada movimiento el que se compara con el mínimo."
assert fecha_recuperacion == "2026-09-05", \
    "fecha_recuperacion: es la primera vez que el stock pasa de estar bajo el mínimo a superarlo (necesitas el stock anterior)."
assert mayor_salida is not None and mayor_salida[1] == "salida", \
    "mayor_salida debe ser una salida: ¿incluiste las entradas (como la de 30 unidades)?"
assert mayor_salida == ("2026-09-06", "salida", 15), \
    "mayor_salida debe ser la tupla completa de la primera salida de 15: ¿usaste >= y te quedaste con el último empate?"
assert movimientos_kardex == _kardex_original, "Cuidado: modificaste la lista original `movimientos_kardex`."

print("✅ Ejercicio 2.4 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Lleva estas variables de estado: el stock (empieza en stock_inicial), la lista de fechas, la fecha de recuperación (empieza en None), y la mayor salida (empieza en None). Para la recuperación necesitas también el stock anterior.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('En cada vuelta: guarda stock_anterior = stock; suma o resta según el tipo; si stock < stock_minimo, agrega la fecha; si fecha_recuperacion es None y stock_anterior < stock_minimo y stock > stock_minimo, guarda la fecha. La mayor salida se actualiza solo dentro de las salidas, con mayor_salida is None or cantidad > mayor_salida[2].')

## 🔒 Soluciones del módulo 2
Abre cada celda **solo después de intentarlo**.

In [ ]:
# @title 🔒 Solución 2.1 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Arrancar saldo en 0 y sumar saldo_inicial al final: saldo_final sale bien pero cada saldo intermedio está desplazado.
# Filtrar el monto 0 elimina un día de la lista aunque el saldo de ese día existe.
# Guardar saldo sin round deja valores como 164.70000000000002.

saldo_inicial = 250.0
movimientos = [
    ("2026-06-01", "proveedor", -180.50),
    ("2026-06-02", "ventas del día", 95.20),
    ("2026-06-03", "ajuste", 0.0),
    ("2026-06-04", "luz", -64.30),
    ("2026-06-05", "ventas del día", 132.75),
    ("2026-06-06", "agua", -28.10),
    ("2026-06-07", "proveedor", -90.00),
    ("2026-06-08", "ventas del día", 210.40),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

saldo = saldo_inicial            # el estado arranca en el valor inicial
saldos = []
for fecha, concepto, monto in movimientos:
    saldo += monto               # incluye los montos 0: no hace falta filtrarlos
    saldos.append(round(saldo, 2))
saldo_final = round(saldo, 2)

In [ ]:
# @title 🔒 Solución 2.2 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Sin break el bucle sigue y se queda con el último día en rojo en vez del primero.
# Inicializar primer_dia con "" o 0 en lugar de None: cuando nunca hay sobregiro entregas un dato falso.
# Olvidar reiniciar saldo antes de la segunda cuenta arrastra el saldo de la primera.

saldo_inicial_a = 120.0
movimientos_a = [
    ("2026-07-01", "proveedor", -75.00),
    ("2026-07-02", "ventas", 60.00),
    ("2026-07-03", "luz", -50.00),
    ("2026-07-04", "ajuste", 0.00),
    ("2026-07-05", "proveedor", -98.50),
    ("2026-07-06", "ventas", 40.00),
    ("2026-07-07", "ventas", 70.00),
]

saldo_inicial_b = 500.0
movimientos_b = [
    ("2026-07-01", "proveedor", -120.00),
    ("2026-07-02", "ventas", 85.50),
    ("2026-07-03", "agua", -31.20),
    ("2026-07-04", "luz", -58.90),
    ("2026-07-05", "ventas", 60.00),
    ("2026-07-06", "proveedor", -140.25),
]
_originales = (list(movimientos_a), list(movimientos_b))   # copias para la validación (no las toques)

saldo = saldo_inicial_a
primer_dia_a = None                      # centinela: todavía no hay sobregiro
for fecha, concepto, monto in movimientos_a:
    saldo += monto
    if saldo < 0:
        primer_dia_a = fecha
        break                            # ya tengo la respuesta: salgo del bucle

saldo = saldo_inicial_b                  # el estado se reinicia con SU valor inicial
primer_dia_b = None
for fecha, concepto, monto in movimientos_b:
    saldo += monto
    if saldo < 0:
        primer_dia_b = fecha
        break

In [ ]:
# @title 🔒 Solución 2.3 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Inicializar el mínimo en 0 solo funciona si hay saldos negativos; con saldos positivos el 0 nunca se supera.
# Con `<=` en lugar de `<` el empate lo gana el último día (2026-08-06) y no el primero.
# Guardar solo el valor y no la fecha obliga a recorrer otra vez para encontrarla.

saldo_inicial = 400.0
movimientos = [
    ("2026-08-01", "proveedor", -150.00),
    ("2026-08-02", "ventas", 80.00),
    ("2026-08-03", "luz", -95.50),
    ("2026-08-04", "ventas", 120.30),
    ("2026-08-05", "alquiler", -200.00),
    ("2026-08-06", "ajuste", 0.00),
    ("2026-08-07", "ventas", 99.00),
    ("2026-08-08", "agua", -44.10),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

saldo = saldo_inicial
saldo_minimo = None                                   # centinela: aún no hay mínimo
fecha_saldo_minimo = None
for fecha, concepto, monto in movimientos:
    saldo += monto
    if saldo_minimo is None or saldo < saldo_minimo:  # `<` conserva el primero en un empate
        saldo_minimo = saldo
        fecha_saldo_minimo = fecha
saldo_minimo = round(saldo_minimo, 2)

In [ ]:
# @title 🔒 Solución 2.4 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Comparar con el stock mínimo sin recordar el stock anterior: no distingues «estar bajo» de «volver a superarlo».
# Buscar la mayor cantidad entre todos los movimientos incluye entradas (la de 30 unidades).
# Con `>=` en vez de `>` el empate lo gana la última salida de 15.

stock_inicial = 40
stock_minimo = 15
movimientos_kardex = [
    ("2026-09-01", "salida", 12),
    ("2026-09-02", "salida", 9),
    ("2026-09-03", "salida", 7),
    ("2026-09-04", "salida", 4),
    ("2026-09-05", "entrada", 20),
    ("2026-09-06", "salida", 15),
    ("2026-09-07", "salida", 3),
    ("2026-09-08", "entrada", 30),
    ("2026-09-09", "salida", 15),
    ("2026-09-10", "salida", 15),
]
_kardex_original = list(movimientos_kardex)   # copia para la validación (no la toques)

stock = stock_inicial
fechas_bajo_minimo = []
fecha_recuperacion = None
mayor_salida = None
for movimiento in movimientos_kardex:            # un solo recorrido, cuatro resultados
    fecha, tipo, cantidad = movimiento
    stock_anterior = stock
    if tipo == "entrada":
        stock += cantidad
    else:
        stock -= cantidad
        if mayor_salida is None or cantidad > mayor_salida[2]:   # `>` estricto: gana el primero
            mayor_salida = movimiento                            # registro completo
    if stock < stock_minimo:
        fechas_bajo_minimo.append(fecha)
    if fecha_recuperacion is None and stock_anterior < stock_minimo and stock > stock_minimo:
        fecha_recuperacion = fecha
stock_final = stock

# Módulo 3 — Comparación de tuplas y parámetro `key`
⏱️ **50 min** · Objetivo: saber **qué compara Python** cuando le pides `min`, `max` o `sorted` sobre tuplas, y decírselo con `key` para que compare lo que tú necesitas.

## 1. Cómo compara Python dos tuplas

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 → 
#   línea 2 → 
#   línea 3 → 
#   línea 4 → 

print(('2026-09-10', -5) < ('2026-09-02', -500))      # línea 1
print((3, 'b') < (3, 'c'))                            # línea 2
print((2, 100) < (10, 0))                             # línea 3
print((5, 0, 1) < (5, 0))                             # línea 4

**Teoría corta**

Python compara tuplas **elemento por elemento, de izquierda a derecha**:

1. Compara el índice 0. Si son **distintos**, ahí se decide y **nada más se mira**.
2. Solo si **empatan**, pasa al índice 1, y así sucesivamente.
3. Si todo lo comparado es igual, la tupla **más corta** es la menor.

Por eso `('2026-09-10', -5) < ('2026-09-02', -500)` es `False`: la fecha ya decide, y el monto ni se mira.

In [ ]:
a = ('2026-09-10', -5)
b = ('2026-09-02', -500)

for i in range(min(len(a), len(b))):
    if a[i] == b[i]:
        print(f"índice {i}: {a[i]!r} == {b[i]!r} → empate, sigo con el siguiente")
    else:
        print(f"índice {i}: {a[i]!r} vs {b[i]!r} → decide aquí: a < b es {a[i] < b[i]}")
        break

print("resultado completo:", a < b)

## 2. Comparar textos: `'10' < '9'` y las fechas

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 → 
#   línea 2 → 
#   línea 3 → 
#   línea 4 → 
#   línea 5 → 

print('10' < '9')                                                 # línea 1
print('2026-09-10' < '2026-09-02')                                # línea 2
print('2026-9-10' < '2026-9-2')                                   # línea 3
print(sorted(['2026-10-01', '2026-09-15', '2026-09-02']))        # línea 4
print(sorted(['2026-9-10', '2026-9-2', '2026-10-1']))            # línea 5

**Teoría corta**

- Los textos se comparan **carácter por carácter**, no como números: `'10' < '9'` es `True` porque `'1'` va antes que `'9'`.
- Las fechas ISO **con ceros** (`AAAA-MM-DD`, todas del mismo largo) **ordenan bien como texto**: comparar textos equivale a comparar fechas.
- **Sin ceros** (`'2026-9-10'`) el orden se rompe. Solución: convertir cada parte a `int` y comparar `(año, mes, día)` como tupla de números.

In [ ]:
con_ceros = ["2026-10-01", "2026-09-15", "2026-09-02"]
sin_ceros = ["2026-9-15", "2026-10-1", "2026-9-2"]

print("con ceros, ordenadas como texto:", sorted(con_ceros))
print("sin ceros, ordenadas como texto:", sorted(sin_ceros), "← orden roto")

convertidas = []
for texto in sin_ceros:
    anio, mes, dia = texto.split("-")
    convertidas.append((int(anio), int(mes), int(dia)))
print("sin ceros, como tuplas de números:", sorted(convertidas), "← orden correcto")

## 3. La trampa: `min(lista_de_tuplas)`

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 (min de la lista) → 
#   línea 2 (max de la lista) → 

movimientos = [
    ("2026-04-01", "mercado", -45.20),
    ("2026-04-06", "sueldo", 1800.00),
    ("2026-04-09", "alquiler", -650.00),
    ("2026-04-14", "yape", 230.00),
    ("2026-04-20", "luz", -96.40),
]
print(min(movimientos))          # línea 1
print(max(movimientos))          # línea 2

**Teoría corta**

`min(movimientos)` compara **tuplas completas**, así que decide por el **índice 0: la fecha**. Devuelve el movimiento más **antiguo**, no el de **menor monto**. Con `max` pasa lo mismo: el más **reciente**.

Es una trampa silenciosa: no da error, da un resultado con pinta razonable pero incorrecto. Si tu dato interesante no está en el índice 0, necesitas decirle a Python **qué comparar**: eso es `key`.

In [ ]:
movimientos = [
    ("2026-04-01", "mercado", -45.20),
    ("2026-04-06", "sueldo", 1800.00),
    ("2026-04-09", "alquiler", -650.00),
    ("2026-04-14", "yape", 230.00),
    ("2026-04-20", "luz", -96.40),
]

print("min sin key      :", min(movimientos), "← el más ANTIGUO")

# Módulo 2 (a mano): centinela None + registro completo
mejor_registro = None
for registro in movimientos:
    if mejor_registro is None or registro[2] < mejor_registro[2]:
        mejor_registro = registro
print("a mano por monto :", mejor_registro)

# Lo mismo con key (siguiente sección)
print("min con key      :", min(movimientos, key=lambda m: m[2]))

## 4. El parámetro `key` en `min`, `max` y `sorted`

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 → 
#   línea 2 → 
#   línea 3 (primeros 2 elementos) → 
#   línea 4 → 
#   línea 5 → 

movs = [("2026-03-09", "internet", -80.0), ("2026-03-01", "luz", -45.0),
        ("2026-03-05", "mercado", -80.0), ("2026-03-12", "yape", 150.0)]

print(min(movs, key=lambda m: m[2]))                             # línea 1
print(max(movs, key=lambda m: m[2]))                             # línea 2
print(sorted(movs, key=lambda m: (m[2], m[0]))[:2])              # línea 3

gasto = {"abarrotes": 320.5, "lácteos": 185.0, "bebidas": 402.25}
print(max(gasto, key=gasto.get))                                 # línea 4
print(max(gasto.items(), key=lambda kv: kv[1]))                  # línea 5

**Teoría corta**

- `key` recibe una función que Python aplica a **cada elemento** para obtener **lo que se compara**. El elemento devuelto es el original, no la clave.
- `lambda m: m[2]` es una función pequeña sin nombre: «dado `m`, devuelve `m[2]`» (el monto).
- Sirve igual en `min`, `max` y `sorted`. En `sorted`, `reverse=True` ordena de mayor a menor.
- **Desempate:** haz que `key` devuelva una **tupla**. `lambda m: (-m[2], m[0])` ordena por monto **descendente** (el signo menos invierte) y, si empatan, por fecha **ascendente**.
- Si dos elementos empatan en la clave, `min` y `max` devuelven **el primero** que aparece.
- Diccionarios: `max(d, key=d.get)` devuelve la **clave** con mayor valor; `max(d.items(), key=lambda kv: kv[1])` devuelve el **par** `(clave, valor)`.

In [ ]:
ventas = [("2026-03-09", "abarrotes", 90.0), ("2026-03-01", "lácteos", 120.0),
          ("2026-03-05", "bebidas", 120.0), ("2026-03-12", "snacks", 45.0)]

print("mayor monto (empate → el primero):", max(ventas, key=lambda v: v[2]))
print("menor monto:", min(ventas, key=lambda v: v[2]))

print("\nranking de mayor a menor, desempate por fecha más antigua:")
for posicion, venta in enumerate(sorted(ventas, key=lambda v: (-v[2], v[0])), start=1):
    print(f"  {posicion}. {venta}")

print("\nmás reciente:", max(ventas, key=lambda v: v[0]))

## 5. `zip` y `enumerate`

In [ ]:
# 🔮 Predice antes de ejecutar
# Escribe tu predicción aquí, ANTES de ejecutar:
#   línea 1 → 
#   línea 2 → 
#   línea 3, 4 y 5 (el for) → 

fechas = ["2026-05-01", "2026-05-02", "2026-05-03"]
ventas = [120.0, 95.5, 210.0, 80.0]

print(list(zip(fechas, ventas)))                                 # línea 1
print(list(enumerate(fechas, start=1)))                          # línea 2
for numero, (fecha, venta) in enumerate(zip(fechas, ventas), start=1):
    print(numero, fecha, venta)                                  # líneas 3 a 5

**Teoría corta**

- **`zip(a, b)`** recorre **listas paralelas** a la vez, emparejando por posición: `(a[0], b[0])`, `(a[1], b[1])`… **Se corta en la más corta**: los elementos sobrantes se ignoran sin avisar.
- **`enumerate(a, start=1)`** te da la **posición** junto al elemento: `(1, a[0])`, `(2, a[1])`… Úsalo cuando necesitas el número de orden («día 3»).
- **Combinados:** `for numero, (x, y) in enumerate(zip(a, b), start=1):` — el par va entre paréntesis porque `zip` entrega una tupla dentro de la tupla de `enumerate`.

In [ ]:
dias = ["lunes", "martes", "miércoles"]
ventas = [120.0, 95.5, 210.0, 80.0]      # ¡una más que los días!

print("zip se corta en la más corta:", len(list(zip(dias, ventas))), "pares de", len(ventas), "ventas")

mejor_numero, mejor_dia, mejor_venta = None, None, None
for numero, (dia, venta) in enumerate(zip(dias, ventas), start=1):
    print(f"día {numero}: {dia:<10} S/ {venta:>7.2f}")
    if mejor_venta is None or venta > mejor_venta:
        mejor_numero, mejor_dia, mejor_venta = numero, dia, venta
print(f"\nmejor: día {mejor_numero} ({mejor_dia}) con S/ {mejor_venta:.2f}")

## 🏋️ Ejercicios del módulo 3

### 🟢 Básico Ejercicio 3.1 — Predicciones evaluadas

Escribe **tu predicción** para cada expresión **sin ejecutarla** (`True` o `False`; en la última, la tupla que devuelve `min`). La validación evalúa las expresiones reales y compara con lo que escribiste.

**Variables a crear**

| Nombre | Tipo | Expresión a predecir |
|---|---|---|
| `pred_a` | `bool` | `('2026-11-03', 10) < ('2026-11-03', 9)` |
| `pred_b` | `bool` | `('b', 1) < ('a', 99)` |
| `pred_c` | `bool` | `(1, 500) < (1, 1000)` |
| `pred_d` | `bool` | `('2026-10-5', 0) < ('2026-10-15', 0)` |
| `pred_e` | `bool` | `'100' < '25'` |
| `pred_f` | `bool` | `(7, 3) < (7, 3, 0)` |
| `pred_min` | `tuple` | `min(lista_prueba)` |

In [ ]:
lista_prueba = [("2026-05-01", 30.0), ("2026-04-20", 90.0), ("2026-05-01", 25.0),
                ("2026-04-27", 15.5), ("2026-05-09", 12.0), ("2026-04-20", 40.0)]

# Escribe True o False (sin ejecutar la expresión):
pred_a = None   # ('2026-11-03', 10) < ('2026-11-03', 9)
pred_b = None   # ('b', 1) < ('a', 99)
pred_c = None   # (1, 500) < (1, 1000)
pred_d = None   # ('2026-10-5', 0) < ('2026-10-15', 0)
pred_e = None   # '100' < '25'
pred_f = None   # (7, 3) < (7, 3, 0)
pred_min = None # la tupla que devuelve min(lista_prueba)

In [ ]:
assert pred_a is (('2026-11-03', 10) < ('2026-11-03', 9)), \
    "pred_a: las fechas empatan en el índice 0, así que decide el índice 1 (¿10 < 9?)."
assert pred_b is (('b', 1) < ('a', 99)), \
    "pred_b: se decide en el índice 0 ('b' vs 'a'); el 99 no se mira."
assert pred_c is ((1, 500) < (1, 1000)), \
    "pred_c: empatan en el índice 0; decide 500 < 1000 (son números, no textos)."
assert pred_d is (('2026-10-5', 0) < ('2026-10-15', 0)), \
    "pred_d: sin cero, se comparan los caracteres '5' y '1'; los textos no se comparan como números."
assert pred_e is ('100' < '25'), \
    "pred_e: son textos, se compara carácter por carácter: '1' contra '2'."
assert pred_f is ((7, 3) < (7, 3, 0)), \
    "pred_f: todo lo comparado empata; la tupla más corta es la menor."
assert pred_min == min(lista_prueba), \
    "pred_min: min compara la tupla completa, empezando por el índice 0 (la fecha), no por el monto."

print("✅ Ejercicio 3.1 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('En cada tupla, mira SOLO el índice 0. Si son distintos, ahí se decide y el resto no importa. Si empatan, pasa al índice 1.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print("Para los textos, compara carácter por carácter desde la izquierda: '100' vs '25' se decide en el primer carácter. Para min(lista_prueba), busca la fecha (índice 0) más antigua y, si hay empate de fecha, mira el monto.")

### 🟡 Intermedio Ejercicio 3.2 — Mayor ingreso y mayor egreso con `key`

Con los movimientos de abril `(fecha, concepto, monto)`, obtén el **movimiento completo** de mayor ingreso y el de mayor egreso (el monto más negativo) usando `max` y `min` con `key`.

**Variables a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `mayor_ingreso` | `tuple` | movimiento con el monto más alto |
| `mayor_egreso` | `tuple` | movimiento con el monto más bajo (más negativo) |

In [ ]:
movimientos = [
    ("2026-04-01", "mercado", -45.20),
    ("2026-04-03", "yape", 230.00),
    ("2026-04-06", "sueldo", 1800.00),
    ("2026-04-08", "alquiler", -650.00),
    ("2026-04-10", "transporte", -58.00),
    ("2026-04-13", "luz", -96.40),
    ("2026-04-17", "yape", 415.50),
    ("2026-04-19", "internet", -79.90),
    ("2026-04-22", "ajuste", 0.00),
    ("2026-04-25", "mercado", -188.75),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

mayor_ingreso = None   # tuple (fecha, concepto, monto)
mayor_egreso = None    # tuple (fecha, concepto, monto)

# Tu código aquí

In [ ]:
assert isinstance(mayor_ingreso, tuple) and isinstance(mayor_egreso, tuple), \
    "Ambos resultados deben ser tuplas completas (fecha, concepto, monto), no solo el monto."
assert mayor_ingreso == ("2026-04-06", "sueldo", 1800.0), \
    "mayor_ingreso incorrecto: ¿estás comparando por la fecha en vez del monto? (max sin key devuelve la fecha más reciente)"
assert mayor_egreso == ("2026-04-08", "alquiler", -650.0), \
    "mayor_egreso incorrecto: ¿usaste min(movimientos) sin key? Eso devuelve la fecha más antigua, no el monto más bajo."
assert movimientos == _movimientos_originales, "Cuidado: modificaste la lista original `movimientos`."

print("✅ Ejercicio 3.2 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Tanto max como min aceptan key=...: una función que recibe un movimiento y devuelve LO QUE SE COMPARA. Aquí, el monto.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('mayor_ingreso = max(movimientos, key=lambda m: m[2]) y mayor_egreso = min(movimientos, key=lambda m: m[2]). El mayor egreso es el monto MÁS NEGATIVO, o sea el mínimo.')

### 🐛 Ejercicio 3.3 — Top 3 de egresos (encuentra y arregla el bug)

De los movimientos de mayo `(fecha, concepto, monto)`, se quiere el **top 3 de egresos**: los tres egresos más grandes (montos más negativos), **ordenados del mayor al menor egreso**, como lista de tuplas completas.

El código siguiente **corre sin errores pero da un resultado incorrecto**. Encuentra el error y arréglalo.

**Variable a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `top3_egresos` | `list` de `tuple` | los 3 egresos más grandes, del más negativo al menos negativo |

In [ ]:
movimientos = [
    ("2026-05-02", "mercado", -73.60),
    ("2026-05-03", "yape", 180.00),
    ("2026-05-05", "alquiler", -520.00),
    ("2026-05-07", "luz", -88.15),
    ("2026-05-09", "sueldo", 1400.00),
    ("2026-05-11", "internet", -79.90),
    ("2026-05-14", "transporte", -32.40),
    ("2026-05-18", "proveedor", -245.30),
    ("2026-05-21", "mercado", -110.25),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

# 🐛 Este código tiene un error típico. Arréglalo (puedes reescribirlo).
egresos = []
for movimiento in movimientos:
    if movimiento[2] < 0:
        egresos.append(movimiento)

top3_egresos = sorted(egresos)[:3]

In [ ]:
assert isinstance(top3_egresos, list) and len(top3_egresos) == 3, "top3_egresos debe ser una lista de 3 movimientos."
assert all(m[2] < 0 for m in top3_egresos), "¿Incluiste ingresos? Solo cuentan los montos negativos."
assert top3_egresos[0] == ("2026-05-05", "alquiler", -520.0), \
    "El primero debe ser el egreso más grande (alquiler). ¿Estás ordenando por la fecha en vez del monto? sorted sin key compara la tupla completa."
assert top3_egresos == [("2026-05-05", "alquiler", -520.0),
                        ("2026-05-18", "proveedor", -245.3),
                        ("2026-05-21", "mercado", -110.25)], \
    "El top 3 no coincide: ordena por el monto. Con negativos, el mayor egreso es el número MÁS BAJO (cuidado con reverse=True)."
assert movimientos == _movimientos_originales, "Cuidado: modificaste la lista original `movimientos`."

print("✅ Ejercicio 3.3 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('sorted(egresos) sin key compara tuplas completas: ¿por qué campo ordena primero? Mira el índice 0 de cada movimiento.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('Dile a sorted qué comparar: key=lambda m: m[2]. Como los egresos son negativos, el orden ascendente ya deja primero el más negativo (no uses reverse=True). Luego corta con [:3].')

### 🔴 Integrador Ejercicio 3.4 — Ranking de días con desempate

Un minimarket registró en tres listas paralelas las ventas y los clientes de una semana. Con `zip` y `enumerate(..., start=1)` arma un **ranking de días**:

- Ordena de **mayor a menor venta**.
- Si dos días empatan en venta, va primero el que tuvo **menos clientes**.
- Cada elemento del ranking es la tupla `(numero_dia, fecha, venta, clientes)`, donde `numero_dia` es la posición del día en la semana (el primero es 1).

**Variables a crear**

| Nombre | Tipo | Contenido |
|---|---|---|
| `ranking` | `list` de `tuple` | los 7 días ordenados como se indica |
| `dia_ganador` | `int` | número de día (1 a 7) del primer lugar |
| `fecha_ganadora` | `str` | fecha del primer lugar |

In [ ]:
fechas = ["2026-10-05", "2026-10-06", "2026-10-07", "2026-10-08", "2026-10-09", "2026-10-10", "2026-10-11"]
ventas = [412.50, 388.00, 455.20, 455.20, 301.75, 520.60, 520.60]
clientes = [96, 90, 101, 88, 70, 115, 104]
_originales = (list(fechas), list(ventas), list(clientes))   # copias para la validación (no las toques)

ranking = None          # list de tuplas (numero_dia, fecha, venta, clientes)
dia_ganador = None      # int
fecha_ganadora = None   # str

# Tu código aquí

In [ ]:
esperado = [
    (7, "2026-10-11", 520.6, 104),
    (6, "2026-10-10", 520.6, 115),
    (4, "2026-10-08", 455.2, 88),
    (3, "2026-10-07", 455.2, 101),
    (1, "2026-10-05", 412.5, 96),
    (2, "2026-10-06", 388.0, 90),
    (5, "2026-10-09", 301.75, 70),
]
assert isinstance(ranking, list) and len(ranking) == 7, \
    "ranking debe ser una lista de 7 tuplas: ¿zip se cortó por listas de distinto largo?"
assert all(isinstance(r, tuple) and len(r) == 4 for r in ranking), \
    "Cada elemento debe ser (numero_dia, fecha, venta, clientes)."
assert sorted(r[0] for r in ranking) == [1, 2, 3, 4, 5, 6, 7], \
    "Los números de día deben ir del 1 al 7: ¿usaste enumerate(..., start=1)?"
assert ranking[0][2] == 520.6 and ranking[-1][2] == 301.75, \
    "El ranking debe ir de MAYOR a MENOR venta: ¿ordenaste al revés?"
assert ranking == esperado, \
    "El ranking no coincide: en empate de venta va primero el de MENOS clientes. Prueba key=lambda r: (-r[2], r[3])."
assert dia_ganador == 7, \
    "dia_ganador debe ser 7: hay un empate en la venta más alta y gana el día con menos clientes."
assert fecha_ganadora == "2026-10-11", "fecha_ganadora debe salir del primer lugar del ranking."
assert (fechas, ventas, clientes) == _originales, "Cuidado: modificaste alguna de las listas originales."

print("✅ Ejercicio 3.4 superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Primero une las tres listas en una sola lista de tuplas, agregando el número de día: for numero, (fecha, venta, cli) in enumerate(zip(fechas, ventas, clientes), start=1). Después ordena esa lista.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('Ordena con sorted(registros, key=lambda r: (-r[2], r[3])): el signo menos invierte el orden de la venta y r[3] desempata por clientes. El ganador es ranking[0].')

## 🔒 Soluciones del módulo 3
Abre cada celda **solo después de intentarlo**.

In [ ]:
# @title 🔒 Solución 3.1 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Leer los textos como números: '100' < '25' es True porque se compara '1' con '2'.
# Mirar el monto cuando la fecha ya decidió: el índice 1 solo entra si el índice 0 empata.
# Creer que min(lista_prueba) devuelve el menor monto: devuelve la fecha más antigua (y, si empatan, el menor monto de esas).

lista_prueba = [("2026-05-01", 30.0), ("2026-04-20", 90.0), ("2026-05-01", 25.0),
                ("2026-04-27", 15.5), ("2026-05-09", 12.0), ("2026-04-20", 40.0)]

pred_a = False   # empatan en la fecha; 10 < 9 es False
pred_b = False   # 'b' < 'a' es False: decide el índice 0
pred_c = True    # empatan en 1; 500 < 1000
pred_d = False   # '5' < '1' es False: comparación de caracteres
pred_e = True    # '1' < '2'
pred_f = True    # todo empata: la más corta es menor
pred_min = ("2026-04-20", 40.0)   # fecha más antigua; en el empate de fecha decide el monto (40.0 < 90.0)

In [ ]:
# @title 🔒 Solución 3.2 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# min(movimientos) / max(movimientos) sin key comparan por la fecha (índice 0): devuelven el más antiguo / el más reciente.
# Usar max para el egreso: el monto más grande en valor absoluto es el MÁS BAJO, o sea el mínimo.
# Devolver solo el monto (m[2]) en vez del movimiento completo.

movimientos = [
    ("2026-04-01", "mercado", -45.20),
    ("2026-04-03", "yape", 230.00),
    ("2026-04-06", "sueldo", 1800.00),
    ("2026-04-08", "alquiler", -650.00),
    ("2026-04-10", "transporte", -58.00),
    ("2026-04-13", "luz", -96.40),
    ("2026-04-17", "yape", 415.50),
    ("2026-04-19", "internet", -79.90),
    ("2026-04-22", "ajuste", 0.00),
    ("2026-04-25", "mercado", -188.75),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

mayor_ingreso = max(movimientos, key=lambda m: m[2])   # compara por el monto (índice 2)
mayor_egreso = min(movimientos, key=lambda m: m[2])    # el egreso más grande es el monto más negativo

In [ ]:
# @title 🔒 Solución 3.3 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# sorted(lista_de_tuplas) sin key ordena por la fecha: el «top 3» resulta ser los tres egresos más antiguos.
# Con montos negativos, reverse=True deja PRIMERO el egreso más pequeño; el orden ascendente es el correcto.
# Olvidar filtrar los ingresos mete montos positivos al ranking.

movimientos = [
    ("2026-05-02", "mercado", -73.60),
    ("2026-05-03", "yape", 180.00),
    ("2026-05-05", "alquiler", -520.00),
    ("2026-05-07", "luz", -88.15),
    ("2026-05-09", "sueldo", 1400.00),
    ("2026-05-11", "internet", -79.90),
    ("2026-05-14", "transporte", -32.40),
    ("2026-05-18", "proveedor", -245.30),
    ("2026-05-21", "mercado", -110.25),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

egresos = []
for movimiento in movimientos:
    if movimiento[2] < 0:
        egresos.append(movimiento)

top3_egresos = sorted(egresos, key=lambda m: m[2])[:3]   # por monto, ascendente: el más negativo primero

In [ ]:
# @title 🔒 Solución 3.4 (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Ordenar solo por venta con reverse=True no desempata: los empates quedan en el orden original (gana el día 6 en vez del 7).
# Poner el signo menos en el desempate equivocado: -r[3] daría prioridad a MÁS clientes.
# Usar enumerate sin start=1: el día ganador saldría 6 en vez de 7.

fechas = ["2026-10-05", "2026-10-06", "2026-10-07", "2026-10-08", "2026-10-09", "2026-10-10", "2026-10-11"]
ventas = [412.50, 388.00, 455.20, 455.20, 301.75, 520.60, 520.60]
clientes = [96, 90, 101, 88, 70, 115, 104]
_originales = (list(fechas), list(ventas), list(clientes))   # copias para la validación (no las toques)

registros = []
for numero, (fecha, venta, cantidad_clientes) in enumerate(zip(fechas, ventas, clientes), start=1):
    registros.append((numero, fecha, venta, cantidad_clientes))

# venta descendente (-r[2]); si empatan, menos clientes primero (r[3])
ranking = sorted(registros, key=lambda r: (-r[2], r[3]))

dia_ganador = ranking[0][0]
fecha_ganadora = ranking[0][1]

# 🏆 Reto final integrador
⏱️ **40 min** · Un solo problema con **todas las trampas del notebook activas**.

Una bodega abre octubre con `saldo_inicial = 300.0` y 16 movimientos `(fecha, concepto, monto)`, con una fecha por movimiento y en orden cronológico. Necesitas un resumen del mes.

Fíjate desde el principio en lo que puede torcerse: un **ajuste de 0** que cae en pleno sobregiro, un **empate** en el mayor egreso, un mayor egreso que **no** es el movimiento más antiguo, conceptos con un solo movimiento y **dos entradas distintas** en sobregiro.

### Parte A — Resuélvelo como quieras
Puedes usar tantos recorridos como necesites. Deja cada resultado en su variable (nombre y tipo exactos):

| Nombre | Tipo | Contenido |
|---|---|---|
| `saldo_final` | `float` | saldo tras el último movimiento, 2 decimales |
| `n_ingresos` | `int` | cantidad de movimientos con monto > 0 |
| `n_egresos` | `int` | cantidad de movimientos con monto < 0 |
| `gasto_por_concepto` | `dict` | concepto → gasto (egresos en **positivo**, 2 decimales); solo conceptos con egresos |
| `mayor_egreso` | `tuple` | el movimiento completo del egreso más grande; **en empate gana el primero** |
| `dias_sobregiro` | `int` | movimientos tras los cuales el saldo quedó **< 0** |
| `veces_en_sobregiro` | `int` | veces que el saldo pasó de **≥ 0** a **< 0** |
| `primer_dia_sobregiro` | `str` o `None` | fecha del primer saldo < 0 (`None` si no hay) |
| `concepto_mas_gasto` | `str` | concepto con mayor gasto total (aquí no hay empate) |

#### 🏆 Reto — Parte A: resuélvelo a tu manera

In [ ]:
saldo_inicial = 300.0
movimientos = [
    ("2026-10-01", "ventas", 180.50),
    ("2026-10-02", "proveedor", -220.30),
    ("2026-10-03", "luz", -85.60),
    ("2026-10-04", "proveedor", -350.00),
    ("2026-10-05", "ajuste", 0.00),
    ("2026-10-06", "ventas", 240.00),
    ("2026-10-07", "transporte", -32.50),
    ("2026-10-08", "alquiler", -350.00),
    ("2026-10-09", "yape", 95.75),
    ("2026-10-10", "ventas", 210.40),
    ("2026-10-11", "ventas", 150.25),
    ("2026-10-12", "internet", -69.90),
    ("2026-10-13", "proveedor", -60.20),
    ("2026-10-14", "yape", 120.00),
    ("2026-10-15", "agua", -28.60),
    ("2026-10-16", "proveedor", -41.15),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

saldo_final = None            # float (2 decimales)
n_ingresos = None             # int: movimientos con monto > 0
n_egresos = None              # int: movimientos con monto < 0
gasto_por_concepto = None     # dict: concepto → gasto (positivo, 2 decimales), solo egresos
mayor_egreso = None           # tuple completa (fecha, concepto, monto); en empate, el primero
dias_sobregiro = None         # int: movimientos tras los cuales el saldo quedó < 0
veces_en_sobregiro = None     # int: cuántas veces el saldo pasó de >= 0 a < 0
primer_dia_sobregiro = None   # str o None
concepto_mas_gasto = None     # str: concepto con mayor gasto total

# Tu código aquí

In [ ]:
import math

esperado_gasto = {"proveedor": 671.65, "luz": 85.60, "transporte": 32.50, "alquiler": 350.00,
                  "internet": 69.90, "agua": 28.60}

assert saldo_final == 58.65, \
    "saldo_final debe ser 58.65: ¿el saldo arranca en saldo_inicial? ¿lo redondeaste a 2 decimales?"
assert n_ingresos == 6, "n_ingresos: cuenta solo los montos > 0 (el ajuste de 0 no es un ingreso)."
assert n_egresos == 9, "n_egresos: cuenta solo los montos < 0 (el ajuste de 0 no es un egreso)."
assert isinstance(gasto_por_concepto, dict), "gasto_por_concepto debe ser un dict."
assert "ajuste" not in gasto_por_concepto and "ventas" not in gasto_por_concepto and "yape" not in gasto_por_concepto, \
    "gasto_por_concepto solo lleva egresos: ¿entró un ingreso o el ajuste de 0?"
assert set(gasto_por_concepto) == set(esperado_gasto), \
    "Los conceptos no coinciden: revisa que estén todos los egresos, incluidos los de un solo movimiento."
for concepto, gasto in esperado_gasto.items():
    assert math.isclose(gasto_por_concepto[concepto], gasto, abs_tol=0.005), \
        f"El gasto de '{concepto}' no coincide: egresos en positivo, todos los movimientos del concepto."
assert all(round(v, 2) == v for v in gasto_por_concepto.values()), "¿Redondeaste el gasto a 2 decimales?"
assert mayor_egreso is not None and mayor_egreso != ("2026-10-01", "ventas", 180.5), \
    "¿Estás comparando por la fecha en vez del monto? min(movimientos) sin key devuelve el más antiguo."
assert mayor_egreso == ("2026-10-04", "proveedor", -350.0), \
    "mayor_egreso: hay un empate en -350; con `<` gana el primero (2026-10-04), con `<=` el último."
assert dias_sobregiro == 5, \
    "dias_sobregiro debe ser 5: ¿un `continue` en el monto 0 te saltó un día que sí estaba en sobregiro?"
assert veces_en_sobregiro == 2, \
    "veces_en_sobregiro debe ser 2: cuenta cruces de >= 0 a < 0, no días (necesitas el saldo anterior)."
assert primer_dia_sobregiro == "2026-10-04", "primer_dia_sobregiro incorrecto: el primer día con saldo < 0."
assert concepto_mas_gasto == "proveedor", \
    "concepto_mas_gasto: compara el gasto TOTAL por concepto, no el egreso más grande de un solo movimiento."
assert movimientos == _movimientos_originales, "Cuidado: modificaste la lista original `movimientos`."

print("✅ Reto Parte A superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Piensa en qué pregunta responde cada resultado: unos son totales (recorrido completo), otros son estados (saldo) y otros extremos (None + registro completo). Puedes usar un for por resultado.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('Trampas: el saldo arranca en saldo_inicial; NO filtres el monto 0 al calcular el saldo (el día en sobregiro existe igual); el mayor egreso se busca con `<` y None; para contar entradas necesitas saldo_anterior, que se actualiza al final de la vuelta; redondea al entregar.')

### Parte B — Ahora, en un solo `for`
Resuelve **el mismo problema** con **un único recorrido** de `movimientos` que lleve todos los acumuladores a la vez (saldo, conteos, gasto por concepto, mayor egreso, sobregiro). Solo lo que dependa de un diccionario **ya armado** (redondeos, el concepto con más gasto) puede ir **después** del `for`.

Las mismas variables, nombre y tipo, y sus propios `assert`.

In [ ]:
saldo_inicial = 300.0
movimientos = [
    ("2026-10-01", "ventas", 180.50),
    ("2026-10-02", "proveedor", -220.30),
    ("2026-10-03", "luz", -85.60),
    ("2026-10-04", "proveedor", -350.00),
    ("2026-10-05", "ajuste", 0.00),
    ("2026-10-06", "ventas", 240.00),
    ("2026-10-07", "transporte", -32.50),
    ("2026-10-08", "alquiler", -350.00),
    ("2026-10-09", "yape", 95.75),
    ("2026-10-10", "ventas", 210.40),
    ("2026-10-11", "ventas", 150.25),
    ("2026-10-12", "internet", -69.90),
    ("2026-10-13", "proveedor", -60.20),
    ("2026-10-14", "yape", 120.00),
    ("2026-10-15", "agua", -28.60),
    ("2026-10-16", "proveedor", -41.15),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

saldo_final = None            # float (2 decimales)
n_ingresos = None             # int: movimientos con monto > 0
n_egresos = None              # int: movimientos con monto < 0
gasto_por_concepto = None     # dict: concepto → gasto (positivo, 2 decimales), solo egresos
mayor_egreso = None           # tuple completa (fecha, concepto, monto); en empate, el primero
dias_sobregiro = None         # int: movimientos tras los cuales el saldo quedó < 0
veces_en_sobregiro = None     # int: cuántas veces el saldo pasó de >= 0 a < 0
primer_dia_sobregiro = None   # str o None
concepto_mas_gasto = None     # str: concepto con mayor gasto total

# Tu código aquí (un solo for sobre movimientos)

In [ ]:
import math

esperado_gasto = {"proveedor": 671.65, "luz": 85.60, "transporte": 32.50, "alquiler": 350.00,
                  "internet": 69.90, "agua": 28.60}

assert saldo_final == 58.65, \
    "saldo_final debe ser 58.65: ¿el saldo arranca en saldo_inicial? ¿lo redondeaste a 2 decimales?"
assert n_ingresos == 6, "n_ingresos: cuenta solo los montos > 0 (el ajuste de 0 no es un ingreso)."
assert n_egresos == 9, "n_egresos: cuenta solo los montos < 0 (el ajuste de 0 no es un egreso)."
assert isinstance(gasto_por_concepto, dict), "gasto_por_concepto debe ser un dict."
assert "ajuste" not in gasto_por_concepto and "ventas" not in gasto_por_concepto and "yape" not in gasto_por_concepto, \
    "gasto_por_concepto solo lleva egresos: ¿entró un ingreso o el ajuste de 0?"
assert set(gasto_por_concepto) == set(esperado_gasto), \
    "Los conceptos no coinciden: revisa que estén todos los egresos, incluidos los de un solo movimiento."
for concepto, gasto in esperado_gasto.items():
    assert math.isclose(gasto_por_concepto[concepto], gasto, abs_tol=0.005), \
        f"El gasto de '{concepto}' no coincide: egresos en positivo, todos los movimientos del concepto."
assert all(round(v, 2) == v for v in gasto_por_concepto.values()), "¿Redondeaste el gasto a 2 decimales?"
assert mayor_egreso is not None and mayor_egreso != ("2026-10-01", "ventas", 180.5), \
    "¿Estás comparando por la fecha en vez del monto? min(movimientos) sin key devuelve el más antiguo."
assert mayor_egreso == ("2026-10-04", "proveedor", -350.0), \
    "mayor_egreso: hay un empate en -350; con `<` gana el primero (2026-10-04), con `<=` el último."
assert dias_sobregiro == 5, \
    "dias_sobregiro debe ser 5: ¿un `continue` en el monto 0 te saltó un día que sí estaba en sobregiro?"
assert veces_en_sobregiro == 2, \
    "veces_en_sobregiro debe ser 2: cuenta cruces de >= 0 a < 0, no días (necesitas el saldo anterior)."
assert primer_dia_sobregiro == "2026-10-04", "primer_dia_sobregiro incorrecto: el primer día con saldo < 0."
assert concepto_mas_gasto == "proveedor", \
    "concepto_mas_gasto: compara el gasto TOTAL por concepto, no el egreso más grande de un solo movimiento."
assert movimientos == _movimientos_originales, "Cuidado: modificaste la lista original `movimientos`."

print("✅ Reto Parte B superado")

In [ ]:
# @title 💡 Pista 1 { display-mode: "form" }
print('Declara todos los acumuladores ANTES del for (saldo, saldo_anterior, contadores, diccionario, mayor_egreso = None, primer_dia_sobregiro = None...). Dentro del for actualizas todos; después del for solo entregas.')

In [ ]:
# @title 💡 Pista 2 { display-mode: "form" }
print('Dentro del for: saldo += monto; if monto > 0 (ingreso) / elif monto < 0 (egreso: gasto y mayor egreso); luego if saldo < 0 (días, entradas con saldo_anterior, primer día); y al FINAL de la vuelta saldo_anterior = saldo. El concepto con más gasto se busca después, sobre el diccionario.')

## 🔒 Soluciones del reto final
Abre cada celda **solo después de intentarlo**.

In [ ]:
# @title 🔒 Solución Reto Parte A (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Empezar el saldo en 0, filtrar el monto 0 con continue antes del chequeo de sobregiro, usar `<=` en el mayor egreso o min(movimientos) sin key.
# Contar días en vez de entradas: veces_en_sobregiro necesita el saldo anterior.
# Olvidar redondear gasto_por_concepto y saldo_final al entregar.

saldo_inicial = 300.0
movimientos = [
    ("2026-10-01", "ventas", 180.50),
    ("2026-10-02", "proveedor", -220.30),
    ("2026-10-03", "luz", -85.60),
    ("2026-10-04", "proveedor", -350.00),
    ("2026-10-05", "ajuste", 0.00),
    ("2026-10-06", "ventas", 240.00),
    ("2026-10-07", "transporte", -32.50),
    ("2026-10-08", "alquiler", -350.00),
    ("2026-10-09", "yape", 95.75),
    ("2026-10-10", "ventas", 210.40),
    ("2026-10-11", "ventas", 150.25),
    ("2026-10-12", "internet", -69.90),
    ("2026-10-13", "proveedor", -60.20),
    ("2026-10-14", "yape", 120.00),
    ("2026-10-15", "agua", -28.60),
    ("2026-10-16", "proveedor", -41.15),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

# --- saldo, conteos y sobregiro (varios recorridos, cada uno con un propósito) ---
saldo = saldo_inicial
for fecha, concepto, monto in movimientos:
    saldo += monto
saldo_final = round(saldo, 2)

n_ingresos = 0
n_egresos = 0
for fecha, concepto, monto in movimientos:
    if monto > 0:
        n_ingresos += 1
    elif monto < 0:
        n_egresos += 1

# --- gasto por concepto: solo egresos, en positivo ---
gasto_por_concepto = {}
for fecha, concepto, monto in movimientos:
    if monto < 0:                                   # excluye ingresos y el ajuste de 0
        gasto_por_concepto[concepto] = gasto_por_concepto.get(concepto, 0) - monto
for concepto in gasto_por_concepto:
    gasto_por_concepto[concepto] = round(gasto_por_concepto[concepto], 2)

# --- mayor egreso: registro completo, `<` conserva el primero en el empate ---
mayor_egreso = None
for movimiento in movimientos:
    if movimiento[2] < 0 and (mayor_egreso is None or movimiento[2] < mayor_egreso[2]):
        mayor_egreso = movimiento

# --- sobregiro: estado, eventos y primer día ---
saldo = saldo_inicial
saldo_anterior = saldo
dias_sobregiro = 0
veces_en_sobregiro = 0
primer_dia_sobregiro = None
for fecha, concepto, monto in movimientos:
    saldo += monto
    if saldo < 0:
        dias_sobregiro += 1                         # estar en sobregiro (incluye el monto 0)
        if saldo_anterior >= 0:
            veces_en_sobregiro += 1                 # entrar en sobregiro
        if primer_dia_sobregiro is None:
            primer_dia_sobregiro = fecha
    saldo_anterior = saldo

# --- concepto con más gasto, buscado a mano ---
concepto_mas_gasto = None
mayor_gasto = None
for concepto, gasto in gasto_por_concepto.items():
    if mayor_gasto is None or gasto > mayor_gasto:
        mayor_gasto = gasto
        concepto_mas_gasto = concepto

In [ ]:
# @title 🔒 Solución Reto Parte B (un solo for) (ábrela después de intentarlo) { display-mode: "form" }
# Error típico:
# Actualizar saldo_anterior antes de usarlo (debe ser lo último de la vuelta).
# Meter el monto 0 en n_ingresos / n_egresos o en el gasto: los ceros no son ingresos ni egresos.
# Buscar el concepto con más gasto mientras el diccionario aún se está acumulando: hazlo cuando el gasto por concepto ya esté completo.

saldo_inicial = 300.0
movimientos = [
    ("2026-10-01", "ventas", 180.50),
    ("2026-10-02", "proveedor", -220.30),
    ("2026-10-03", "luz", -85.60),
    ("2026-10-04", "proveedor", -350.00),
    ("2026-10-05", "ajuste", 0.00),
    ("2026-10-06", "ventas", 240.00),
    ("2026-10-07", "transporte", -32.50),
    ("2026-10-08", "alquiler", -350.00),
    ("2026-10-09", "yape", 95.75),
    ("2026-10-10", "ventas", 210.40),
    ("2026-10-11", "ventas", 150.25),
    ("2026-10-12", "internet", -69.90),
    ("2026-10-13", "proveedor", -60.20),
    ("2026-10-14", "yape", 120.00),
    ("2026-10-15", "agua", -28.60),
    ("2026-10-16", "proveedor", -41.15),
]
_movimientos_originales = list(movimientos)   # copia para la validación (no la toques)

saldo = saldo_inicial
saldo_anterior = saldo
n_ingresos = 0
n_egresos = 0
gasto_por_concepto = {}
mayor_egreso = None
dias_sobregiro = 0
veces_en_sobregiro = 0
primer_dia_sobregiro = None

for movimiento in movimientos:                       # UN solo recorrido, varios acumuladores
    fecha, concepto, monto = movimiento
    saldo += monto

    if monto > 0:
        n_ingresos += 1
    elif monto < 0:
        n_egresos += 1
        gasto_por_concepto[concepto] = gasto_por_concepto.get(concepto, 0) - monto
        if mayor_egreso is None or monto < mayor_egreso[2]:
            mayor_egreso = movimiento

    if saldo < 0:
        dias_sobregiro += 1
        if saldo_anterior >= 0:
            veces_en_sobregiro += 1
        if primer_dia_sobregiro is None:
            primer_dia_sobregiro = fecha
    saldo_anterior = saldo                            # al final de la vuelta

# Al entregar: redondeos y el extremo por concepto (sobre el diccionario ya armado)
saldo_final = round(saldo, 2)
concepto_mas_gasto = None
mayor_gasto = None
for concepto, gasto in gasto_por_concepto.items():
    if mayor_gasto is None or gasto > mayor_gasto:
        mayor_gasto = gasto
        concepto_mas_gasto = concepto
for concepto in gasto_por_concepto:
    gasto_por_concepto[concepto] = round(gasto_por_concepto[concepto], 2)

## 🏁 Cierre

### Checklist de autoevaluación
Marca solo lo que lograrías **sin abrir una pista**:

- ☐ Agrupo por categoría (total y conteo) con `.get` y un solo `for`.
- ☐ Redondeo al entregar y comparo decimales con `round` o `math.isclose`.
- ☐ Arranco el estado en su valor inicial y puedo explicar por qué.
- ☐ Distingo «estar en sobregiro» de «entrar en sobregiro» y sé qué variable me falta para lo segundo.
- ☐ Busco extremos con `None`, guardo el registro completo y decido el empate con `<` o `<=`.
- ☐ Sé cuándo un `continue` es seguro y cuándo se salta un caso válido.
- ☐ Predigo comparaciones de tuplas y de textos (fechas con y sin ceros).
- ☐ Uso `min`/`max`/`sorted` con `key`, incluido `key` que devuelve una tupla.
- ☐ Combino `zip` y `enumerate(..., start=1)`.

### Lo que viene (los mismos patrones con otras herramientas)

| Lo que practicaste hoy | Lo que viene |
|---|---|
| Agrupar en un diccionario (`d[k] = d.get(k, 0) + x`) | `GROUP BY` en SQL · `groupby` en pandas |
| Saldo acumulado paso a paso (running total) | `SUM() OVER (ORDER BY fecha)` en SQL · `cumsum` en pandas |
| `min(registros, key=...)` para hallar la fila del mínimo | `idxmin` en pandas |
| Varios acumuladores en un solo recorrido | `agg` con varias funciones · varias columnas en un `SELECT ... GROUP BY` |

Cuando llegues a esas herramientas, notarás que cada una hace por ti el recorrido que hoy escribiste a mano; entender lo que pasa dentro te permitirá detectar resultados raros.